<a href="https://colab.research.google.com/github/cem8kaya/open5gs-nwdaf/blob/main/5g_ai_lab_daily_ops_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5G AI/ML Lab — Daily Operations, Maintenance & Troubleshooting
Platform: Google Colab → gcloud SSH → GCP VM open5gs-ai-lab
Stack: Open5GS v2.7.6 · UERANSIM · NWDAF (TS 23.288 / TS 29.520)
PLMN: MCC 999 / MNC 70 · Zone: europe-west4-a · Project: g-ai-lab-491619

# 1. Colab Bootstrap

Run these cells first in every new Colab session. Without them, gcloud commands and helper functions will not work.

## 1.1 — GCP Authentication

In [1]:
from google.colab import auth
auth.authenticate_user()

import subprocess, os, json, time
from datetime import datetime

PROJECT_ID = "g-ai-lab-491619"
ZONE       = "europe-west4-a"
VM_NAME    = "open5gs-ai-lab"
NWDAF_PORT = 7779   # ⚠️ NOT 7777 — Open5GS NFs all use 7777; NWDAF uses 7779

print(f"✅ Auth OK | VM: {VM_NAME} | Zone: {ZONE} | NWDAF port: {NWDAF_PORT}")

✅ Auth OK | VM: open5gs-ai-lab | Zone: europe-west4-a | NWDAF port: 7779


## 1.2 — Helper Functions

In [2]:
def ssh_run(cmd, sudo=False):
    """Run a single command on the VM and return stdout."""
    prefix = "sudo " if sudo else ""
    result = subprocess.run(
        ["gcloud", "compute", "ssh", VM_NAME,
         f"--project={PROJECT_ID}", f"--zone={ZONE}",
         "--command", f"{prefix}{cmd}"],
        capture_output=True, text=True
    )
    if result.stdout: print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print(f"[STDERR] {result.stderr[:500]}")
    return result

def ssh_script(script, description=""):
    """Upload and run a multi-line bash script on the VM."""
    if description:
        print(f"▶ {description}")
    tmp = f"/tmp/ops_{int(time.time())}.sh"
    with open(tmp, "w") as f:
        f.write("#!/bin/bash\nset -e\n")
        f.write(script)
    subprocess.run(
        ["gcloud", "compute", "scp", tmp, f"{VM_NAME}:{tmp}",
         f"--project={PROJECT_ID}", f"--zone={ZONE}"],
        capture_output=True
    )
    result = subprocess.run(
        ["gcloud", "compute", "ssh", VM_NAME,
         f"--project={PROJECT_ID}", f"--zone={ZONE}",
         "--command", f"sudo bash {tmp}; rm -f {tmp}"],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print(f"[STDERR] {result.stderr[:800]}")
    os.remove(tmp)
    return result

def scp_get(remote_path, local_path="."):
    """Download a file from the VM to Colab."""
    import os, shutil
    # If local_path looks like a file (has extension), scp via a temp dir
    # to avoid gcloud scp creating local_path/basename instead of local_path.
    if '.' in os.path.basename(local_path):
        os.makedirs(os.path.dirname(local_path) or '.', exist_ok=True)
        tmp_dir = f"/tmp/scp_tmp_{int(time.time())}"
        os.makedirs(tmp_dir, exist_ok=True)
        r = subprocess.run(
            ["gcloud", "compute", "scp",
             f"{VM_NAME}:{remote_path}", tmp_dir,
             f"--project={PROJECT_ID}", f"--zone={ZONE}"],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            fname = os.path.basename(remote_path)
            os.rename(f"{tmp_dir}/{fname}", local_path)
        shutil.rmtree(tmp_dir, ignore_errors=True)
    else:
        r = subprocess.run(
            ["gcloud", "compute", "scp",
             f"{VM_NAME}:{remote_path}", local_path,
             f"--project={PROJECT_ID}", f"--zone={ZONE}"],
            capture_output=True, text=True
        )
    print(f"✅ Downloaded: {local_path}" if r.returncode == 0
          else f"❌ Error: {r.stderr[:200]}")
    return r

def scp_put(local_path, remote_path="/tmp/"):
    """Upload a file from Colab to the VM."""
    r = subprocess.run(
        ["gcloud", "compute", "scp", local_path,
         f"{VM_NAME}:{remote_path}",
         f"--project={PROJECT_ID}", f"--zone={ZONE}"],
        capture_output=True, text=True
    )
    print(f"✅ Uploaded: {remote_path}" if r.returncode == 0
          else f"❌ Error: {r.stderr[:200]}")
    return r

print("✅ Helper functions ready.")

✅ Helper functions ready.


## 1.3 — Colab-side Python Dependencies

In [3]:
%%bash
pip install -q pandas scapy matplotlib seaborn numpy requests PyYAML pymongo scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.4 MB/s eta 0:00:00


## 1.4 — Upgrade/Deploy Native C++ NWDAF
This cell checks if the native C++ NWDAF is already deployed on the VM. If not, it clones the private repository and builds it. It will prompt for your GitHub PAT if a deployment is required.

In [23]:
import getpass
import subprocess

PROJECT_ID = "g-ai-lab-491619"
ZONE = "europe-west4-a"
VM_NAME = "open5gs-ai-lab"

# Check if already deployed
check_cmd = [
    "gcloud", "compute", "ssh", VM_NAME,
    f"--project={PROJECT_ID}", f"--zone={ZONE}",
    "--command=test -f /usr/local/bin/open5gs-nwdafd && echo 'EXISTS' || echo 'MISSING'"
]
print("Checking VM for existing C++ NWDAF installation...")
check_res = subprocess.run(check_cmd, capture_output=True, text=True)

if "EXISTS" in check_res.stdout:
    print("✅ C++ NWDAF is already deployed and installed on the VM.")
    force = input("Do you want to force re-deploy and rebuild from the latest GitHub code? (y/N): ")
    if force.lower() != 'y':
        print("Skipping deployment.")
        github_token = None
    else:
        github_token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")
else:
    print("C++ NWDAF not found. A deployment is required.")
    github_token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")

if github_token:
    repo_url = f"https://{github_token}@github.com/cem8kaya/open5gs-nwdaf.git"

    bash_script = f"""
    set -e
    echo '--> Cloning latest source code from GitHub...'
    rm -rf /tmp/open5gs-nwdaf
    git clone {repo_url} /tmp/open5gs-nwdaf >/dev/null 2>&1
    cd /tmp/open5gs-nwdaf || {{ echo 'Clone failed!'; exit 1; }}

    echo '--> Installing C++ Build Dependencies...'
    sudo DEBIAN_FRONTEND=noninteractive apt-get update >/dev/null
    sudo DEBIAN_FRONTEND=noninteractive apt-get install -y pkg-config build-essential cmake libyaml-cpp-dev libspdlog-dev libsystemd-dev >/dev/null

    echo '--> Compiling Release Build...'
    mkdir -p build-release && cd build-release
    cmake .. -DCMAKE_BUILD_TYPE=Release -DNWDAF_USE_SD_JOURNAL=ON -DNWDAF_BUILD_TESTS=OFF >/dev/null
    make -j$(nproc) >/dev/null

    echo '--> Installing Binary and Service...'
    sudo systemctl stop open5gs-nwdafd || true
    sudo cp open5gs-nwdafd /usr/local/bin/
    sudo chmod 755 /usr/local/bin/open5gs-nwdafd
    cd ..

    sudo rm -f /opt/nwdaf/nwdaf_server.py
    sudo cp systemd/open5gs-nwdafd.service /etc/systemd/system/
    # Patch the service file to remove the broken watchdog timeout
    sudo sed -i '/WatchdogSec=30/d' /etc/systemd/system/open5gs-nwdafd.service

    sudo mkdir -p /opt/nwdaf/models
    sudo chown -R $(whoami):$(whoami) /opt/nwdaf

    sudo systemctl daemon-reload
    sudo systemctl enable open5gs-nwdafd
    sudo systemctl restart open5gs-nwdafd

    EXISTING_UUID=$(grep 'nf_instance_id:' /etc/open5gs/nwdaf.yaml \
        | grep -oE '[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}' \
        | head -1)
    if [ -n "$EXISTING_UUID" ]; then
        STABLE_UUID="$EXISTING_UUID"
        echo "Reusing existing NWDAF instance ID: $STABLE_UUID"
    else
        STABLE_UUID=$(python3 -c "import uuid; print(uuid.uuid4())")
        echo "Generated new NWDAF instance ID: $STABLE_UUID"
    fi
    sudo sed -i "s|^  nf_instance_id:.*$|  nf_instance_id: \"$STABLE_UUID\"|" \
        /etc/open5gs/nwdaf.yaml
    echo "NWDAF instance ID confirmed: $STABLE_UUID"

    echo '✅ C++ NWDAF successfully deployed and running!'
    """

    cmd = [
        "gcloud", "compute", "ssh", VM_NAME,
        f"--project={PROJECT_ID}",
        f"--zone={ZONE}",
        f"--command={bash_script}"
    ]

    print("Executing deployment on VM...")
    result = subprocess.run(cmd, capture_output=True, text=True)

    print(result.stdout)
    if result.returncode != 0:
        print("\n--- Errors / Warnings ---")
        print(result.stderr)
        print("\nDeployment Failed.")
    else:
        print("Deployment Complete.")

Checking VM for existing C++ NWDAF installation...
✅ C++ NWDAF is already deployed and installed on the VM.
Do you want to force re-deploy and rebuild from the latest GitHub code? (y/N): y
Enter your GitHub Personal Access Token (PAT): ··········
Executing deployment on VM...
--> Cloning latest source code from GitHub...
--> Installing C++ Build Dependencies...
--> Compiling Release Build...
--> Installing Binary and Service...
✅ C++ NWDAF successfully deployed and running!

Deployment Complete.


# 2. Daily Health Dashboard


Run this first every day. Expected: 7/7 NFs active, gNB + UE running, at least one PDU session, upf_sessionnbr ≥ 1.

## 2.1 — Full System Health Dashboard

In [24]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '════════════════════════════════════════════════════'
echo '  OPEN5GS SYSTEM HEALTH DASHBOARD'
echo '  '\$(date '+%Y-%m-%d %H:%M:%S')
echo '════════════════════════════════════════════════════'

echo ''
echo '─── 5G Core Network Functions ───────────────────────'
TOTAL=0; ACTIVE=0
for svc in amfd smfd upfd ausfd udmd pcfd nrfd; do
    STATUS=\$(systemctl is-active open5gs-\$svc 2>/dev/null)
    TOTAL=\$(( TOTAL + 1 ))
    [ \"\$STATUS\" = 'active' ] && ACTIVE=\$(( ACTIVE + 1 ))
    ICON=\$([ \"\$STATUS\" = 'active' ] && echo '✅' || echo '❌')
    printf '  %s %-8s : %s\n' \"\$ICON\" \"\$svc\" \"\$STATUS\"
done
echo \"  Summary: \$ACTIVE/\$TOTAL NFs active\"

echo ''
echo '─── NWDAF ────────────────────────────────────────────'
NWDAF_STATUS=\$(systemctl is-active open5gs-nwdafd 2>/dev/null || echo 'not-installed')
ICON=\$([ \"\$NWDAF_STATUS\" = 'active' ] && echo '✅' || echo '⚠️ ')
printf '  %s nwdafd : %s\n' \"\$ICON\" \"\$NWDAF_STATUS\"
[ \"\$NWDAF_STATUS\" = 'active' ] && \
    curl -s --connect-timeout 3 http://localhost:7779/nwdaf-analytics/v1/health 2>/dev/null | \
    python3 -c 'import sys,json; d=json.load(sys.stdin); print(\"  SBI status:\", d.get(\"status\",\"?\"))' \
    2>/dev/null || true

echo ''
echo '─── UERANSIM ─────────────────────────────────────────'
GNB_PID=\$(pgrep -x nr-gnb 2>/dev/null)
UE_PID=\$(pgrep -x nr-ue 2>/dev/null)
[ -n \"\$GNB_PID\" ] && echo '  ✅ gNB  : running (PID='\$GNB_PID')' || echo '  ❌ gNB  : STOPPED'
[ -n \"\$UE_PID\"  ] && echo '  ✅ UE   : running (PID='\$UE_PID')'  || echo '  ❌ UE   : STOPPED'

echo ''
echo '─── PDU Sessions ────────────────────────────────────'
SESS_COUNT=0
for iface in uesimtun0 uesimtun1 uesimtun2 uesimtun3; do
    IP=\$(ip a show \$iface 2>/dev/null | grep 'inet ' | awk '{print \$2}')
    if [ -n \"\$IP\" ]; then
        echo \"  ✅ \$iface : \$IP\"
        SESS_COUNT=\$(( SESS_COUNT + 1 ))
    fi
done
[ \"\$SESS_COUNT\" -eq 0 ] && echo '  ⚠️  No active PDU sessions'

echo ''
echo '─── UPF Metrics ─────────────────────────────────────'
METRICS=\$(curl -s --connect-timeout 3 http://127.0.0.7:9090/metrics 2>/dev/null)
if [ -n \"\$METRICS\" ]; then
    echo \"\$METRICS\" | grep -v '^#' | grep 'upf_' | \
        awk '{printf \"  %-45s %s\n\", \$1, \$2}'
else
    echo '  ❌ Prometheus endpoint unreachable'
fi

echo ''
echo '─── System Resources ────────────────────────────────'
echo '  CPU  :' \$(top -bn1 | grep 'Cpu(s)' | awk '{print \$2+\$4\"%\"}')
echo '  RAM  :' \$(free -h | awk '/^Mem/{print \$3\"/\"\$2}')
echo '  Disk :' \$(df -h / | awk 'NR==2{print \$3\"/\"\$2\" (\"\$5\" used)\"}')

echo ''
echo '─── NAT / Routing ───────────────────────────────────'
echo '  MASQ rules:'
sudo iptables -t nat -L POSTROUTING -n 2>/dev/null | grep MASQ | \
    awk '{printf \"    %s\n\", \$0}'
echo '  IP Forward:' \$(sysctl -n net.ipv4.ip_forward)
echo '════════════════════════════════════════════════════'
"

════════════════════════════════════════════════════
  OPEN5GS SYSTEM HEALTH DASHBOARD
  2026-04-25 21:50:25
════════════════════════════════════════════════════

─── 5G Core Network Functions ───────────────────────
  ✅ amfd     : active
  ✅ smfd     : active
  ✅ upfd     : active
  ✅ ausfd    : active
  ✅ udmd     : active
  ✅ pcfd     : active
  ✅ nrfd     : active
  Summary: 7/7 NFs active

─── NWDAF ────────────────────────────────────────────
  ✅ nwdafd : active
  SBI status: UP

─── UERANSIM ─────────────────────────────────────────
  ✅ gNB  : running (PID=394768)
  ✅ UE   : running (PID=394791)

─── PDU Sessions ────────────────────────────────────
  ✅ uesimtun0 : 10.45.0.2/24

─── UPF Metrics ─────────────────────────────────────
  fivegs_upffunction_upf_sessionnbr             1
  fivegs_upffunction_upf_qosflows{dnn="internet"} 1

─── System Resources ────────────────────────────────
  CPU  : 9.1%
  RAM  : 967Mi/7.8Gi
  Disk : 11G/29G (36% used)

─── NAT / Routing ────────────

# 3. Service Control

Change ACTION to start, stop, restart, or reload.
Change NF_LIST to target specific NFs or use the full list.

## 3.1 — Open5GS NF Service Control

In [25]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
ACTION="restart"   # start / stop / restart / reload
NF_LIST="amfd smfd upfd"
# Full list: "amfd smfd upfd ausfd udmd pcfd nrfd"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    for NF in $NF_LIST; do
        echo -n \"  \$ACTION open5gs-\$NF ... \"
        sudo systemctl $ACTION open5gs-\$NF 2>&1
        STATUS=\$(systemctl is-active open5gs-\$NF)
        echo \"\$STATUS\"
    done
    echo ''
    echo 'Final status:'
    for NF in $NF_LIST; do
        printf '  %-12s: %s\n' \"\$NF\" \"\$(systemctl is-active open5gs-\$NF)\"
    done
"

   open5gs-amfd ... active
   open5gs-smfd ... active
   open5gs-upfd ... active

Final status:
  amfd        : active
  smfd        : active
  upfd        : active


## 3.2 — NWDAF Service Control

In [26]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
ACTION="status"   # start / stop / restart / status

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '=== NWDAF Service: open5gs-nwdafd ==='
sudo systemctl $ACTION open5gs-nwdafd
echo ''
echo 'Recent logs (20 lines):'
sudo journalctl -u open5gs-nwdafd -n 20 --no-pager
"

=== NWDAF Service: open5gs-nwdafd ===
● open5gs-nwdafd.service - Open5GS NWDAF - Network Data Analytics Function (3GPP TS 23.288)
     Loaded: loaded (/etc/systemd/system/open5gs-nwdafd.service; enabled; vendor preset: enabled)
     Active: active (running) since Sat 2026-04-25 21:50:07 UTC; 32s ago
       Docs: https://open5gs.org
             https://github.com/cem8kaya/5g-ai-lab
   Main PID: 401881 (open5gs-nwdafd)
      Tasks: 12 (limit: 9518)
     Memory: 1.8M
        CPU: 420ms
     CGroup: /system.slice/open5gs-nwdafd.service
             └─401881 /usr/local/bin/open5gs-nwdafd --config /etc/open5gs/nwdaf.yaml

Apr 25 21:50:07 open5gs-ai-lab systemd[1]: Starting Open5GS NWDAF - Network Data Analytics Function (3GPP TS 23.288)...
Apr 25 21:50:07 open5gs-ai-lab open5gs-nwdafd[401881]: [2026-04-25 21:50:07.121] [nwdaf] [info] Open5GS NWDAF starting (instance: 198e9234-b849-42a2-9f70-d9a587616c2b)
Apr 25 21:50:07 open5gs-ai-lab open5gs-nwdafd[401881]: [2026-04-25 21:50:07.123] [nwdaf

# 4. UERANSIM Control

## 4.1 — gNB & UE Process Management

In [27]:
# Cell 4.1 — gNB + UE Restart with AMF NG Setup Verification
# Use any time: daily startup, after AMF restart, after any NF restart
# The script detects state and only restarts what is needed.

import subprocess, os, time

PROJECT_ID = "g-ai-lab-491619"
ZONE       = "europe-west4-a"
VM_NAME    = "open5gs-ai-lab"

def ssh_script(script, description=""):
    if description:
        print(f">> {description}")
    tmp_local  = f"/tmp/ops_{int(time.time())}.sh"
    tmp_remote = f"/tmp/ops_{int(time.time())}.sh"
    with open(tmp_local, "w", encoding="utf-8") as f:
        f.write("#!/bin/bash\nset +e\n")
        f.write(script)
    subprocess.run(
        ["gcloud", "compute", "scp", tmp_local,
         f"{VM_NAME}:{tmp_remote}",
         f"--project={PROJECT_ID}", f"--zone={ZONE}"],
        capture_output=True
    )
    result = subprocess.run(
        ["gcloud", "compute", "ssh", VM_NAME,
         f"--project={PROJECT_ID}", f"--zone={ZONE}",
         "--command", f"bash {tmp_remote}; rm -f {tmp_remote}"],
        capture_output=True, text=True, encoding="utf-8"
    )
    print(result.stdout)
    if result.stderr and result.returncode != 0:
        print(f"[STDERR] {result.stderr[:600]}")
    os.remove(tmp_local)
    return result

RESTART_SCRIPT = r"""
NR="/opt/UERANSIM/build"
CFG_GNB="/opt/UERANSIM/config/open5gs-gnb.yaml"
CFG_UE="/opt/UERANSIM/config/open5gs-ue.yaml"

echo "=== [1] AMF NGAP readiness ==="
AMF_OK=0
for i in $(seq 1 10); do
    if ss -lnp 2>/dev/null | grep -q 38412; then
        echo "  AMF listening on :38412"
        AMF_OK=1
        break
    fi
    echo "  [$i/10] waiting for AMF NGAP..."
    sleep 1
done

if [ "$AMF_OK" -eq 0 ]; then
    echo "  AMF not listening. Restarting open5gs-amfd..."
    sudo systemctl restart open5gs-amfd
    sleep 4
    ss -lnp 2>/dev/null | grep 38412 && echo "  AMF now listening" \
        || echo "  ERROR: AMF still not up. Check: journalctl -u open5gs-amfd -n 30"
fi

echo ""
echo "=== [2] Kill stale gNB and UE ==="
# Clean deregistration avoids leaving orphan PDU sessions in SMF/UPF.
# nr-cli deregister sends NAS Deregistration Request; pkill alone does not.
sudo /opt/UERANSIM/build/nr-cli imsi-999700000000001 --exec "deregister normal" 2>/dev/null || true
sleep 1
sudo pkill -f nr-ue  2>/dev/null; sleep 1
sudo pkill -f nr-gnb 2>/dev/null; sleep 2
echo "  Done"

echo ""
echo "=== [3] Start gNB ==="
sudo nohup "$NR/nr-gnb" -c "$CFG_GNB" > /root/gnb.log 2>&1 &
sleep 5

# Verify NG Setup completed by checking AMF received it
GNB_SETUP=$(grep -i "NG Setup procedure is successful" /root/gnb.log 2>/dev/null | tail -1)
if [ -n "$GNB_SETUP" ]; then
    echo "  NG Setup confirmed in gNB log: $GNB_SETUP"
else
    AMF_SETUP=$(sudo journalctl -u open5gs-amfd -n 50 --no-pager 2>/dev/null \
        | grep -iE "NG-Setup|ran_id\[|gNB-ID\[" | tail -3)
    [ -n "$AMF_SETUP" ] && echo "  NG Setup in AMF log: $AMF_SETUP" \
                        || echo "  WARNING: NG Setup not confirmed yet."
fi

echo ""
echo "=== [4] Start UE ==="
sudo nohup "$NR/nr-ue" -c "$CFG_UE" > /root/ue.log 2>&1 &

UE_IFACE=""
echo "  Waiting for PDU session (up to 30s)..."
for i in $(seq 1 60); do
    sleep 0.5
    UE_IFACE=$(ip a 2>/dev/null | grep -o 'uesimtun[0-9]*' | head -1)
    if [ -n "$UE_IFACE" ]; then
        IP=$(ip a show "$UE_IFACE" 2>/dev/null | grep 'inet ' | awk '{print $2}')
        echo "  PDU session up: $UE_IFACE -> $IP"
        break
    fi
done

if [ -z "$UE_IFACE" ]; then
    echo "  ERROR: No tunnel after 30s."
    echo ""
    echo "  -- UE log (last 20 lines) --"
    tail -20 /root/ue.log
    echo ""
    echo "  -- AMF log (last 20 lines) --"
    sudo journalctl -u open5gs-amfd -n 20 --no-pager 2>/dev/null
fi

echo ""
echo "=== [5] Final state ==="
pgrep -x nr-gnb > /dev/null && echo "  gNB: running" || echo "  gNB: STOPPED"
pgrep -x nr-ue  > /dev/null && echo "  UE:  running" || echo "  UE:  STOPPED"
ip a 2>/dev/null | grep -E 'uesimtun|ogstun' \
    | awk '{print "  iface:", $0}'
echo "  UPF sessions:"
curl -s http://127.0.0.7:9090/metrics 2>/dev/null \
    | grep 'upf_sessionnbr' | grep -v '#' \
    | awk '{print "   ", $0}'
"""

ssh_script(RESTART_SCRIPT, "gNB + UE Restart with AMF NG Setup Verification")

>> gNB + UE Restart with AMF NG Setup Verification
=== [1] AMF NGAP readiness ===
  AMF listening on :38412

=== [2] Kill stale gNB and UE ===
  Done

=== [3] Start gNB ===
  gNB log tail:
UERANSIM v3.2.7
[2026-04-25 21:50:57.806] [sctp] [info] Trying to establish SCTP connection... (127.0.0.5:38412)
[2026-04-25 21:50:57.809] [sctp] [info] SCTP connection established (127.0.0.5:38412)
[2026-04-25 21:50:57.810] [sctp] [debug] SCTP association setup ascId[85]
[2026-04-25 21:50:57.810] [ngap] [debug] Sending NG Setup Request
[2026-04-25 21:50:57.820] [ngap] [debug] NG Setup Response received
[2026-04-25 21:50:57.820] [ngap] [info] NG Setup procedure is successful

=== [4] Start UE ===
  Waiting for PDU session (up to 30s)...
  PDU session up: uesimtun0 -> 10.45.0.2/24

=== [5] Final state ===
  gNB: running
  UE:  running
  iface: 3: ogstun: <POINTOPOINT,MULTICAST,NOARP,UP,LOWER_UP> mtu 1400 qdisc fq_codel state UP group default qlen 500
  iface:     inet 10.45.0.1/16 brd 10.45.255.255 sc

CompletedProcess(args=['gcloud', 'compute', 'ssh', 'open5gs-ai-lab', '--project=g-ai-lab-491619', '--zone=europe-west4-a', '--command', 'bash /tmp/ops_1777153840.sh; rm -f /tmp/ops_1777153840.sh'], returncode=0, stdout='=== [1] AMF NGAP readiness ===\n  AMF listening on :38412\n\n=== [2] Kill stale gNB and UE ===\n  Done\n\n=== [3] Start gNB ===\n  WARNING: No NG Setup seen in AMF log yet.\n  gNB log tail:\nUERANSIM v3.2.7\n[2026-04-25 21:50:57.806] [sctp] [info] Trying to establish SCTP connection... (127.0.0.5:38412)\n[2026-04-25 21:50:57.809] [sctp] [info] SCTP connection established (127.0.0.5:38412)\n[2026-04-25 21:50:57.810] [sctp] [debug] SCTP association setup ascId[85]\n[2026-04-25 21:50:57.810] [ngap] [debug] Sending NG Setup Request\n[2026-04-25 21:50:57.820] [ngap] [debug] NG Setup Response received\n[2026-04-25 21:50:57.820] [ngap] [info] NG Setup procedure is successful\n\n=== [4] Start UE ===\n  Waiting for PDU session (up to 30s)...\n  PDU session up: uesimtun0 -> 10.45

## 4.2 — nr-cli UE Session Control

In [28]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    NR='/opt/UERANSIM/build'

    echo '=== Active UERANSIM Nodes ==='
    cd \$NR && ./nr-cli --dump 2>/dev/null || echo 'nr-cli: no UE found'

    echo ''
    echo '=== First UE Status ==='
    UE_NODE=\$(cd \$NR && ./nr-cli --dump 2>/dev/null | grep 'imsi-' | head -1 | awk '{print \$1}')
    if [ -n \"\$UE_NODE\" ]; then
        echo 'Node:' \$UE_NODE
        cd \$NR && echo 'status' | ./nr-cli \"\$UE_NODE\" 2>/dev/null
    else
        echo 'No active UE session'
    fi

    echo ''
    echo '=== Available nr-cli Commands ==='
    echo '  status                - UE state info'
    echo '  deregister normal     - 5G deregistration (~2s clean teardown)'
    echo '  deregister disable-5g - Disable 5G'
    echo '  ps-release 1          - Release PDU session 1'
    echo '  ps-establish default  - Establish new PDU session'
"

=== Active UERANSIM Nodes ===
UERANSIM-gnb-999-70-1
imsi-999700000000001

=== First UE Status ===
Node: imsi-999700000000001
--------------------------------------------------------------------------------------------
$ cm-state: CM-CONNECTED
rm-state: RM-REGISTERED
mm-state: MM-REGISTERED/NORMAL-SERVICE
5u-state: 5U1-UPDATED
sim-inserted: true
selected-plmn: 999/70
current-cell: 1
current-plmn: 999/70
current-tac: 1
last-tai: PLMN[999/70] TAC[1]
stored-suci: no-identity
stored-guti: 
 plmn: 999/70
 amf-region-id: 0x02
 amf-set-id: 1
 amf-pointer: 0
 tmsi: 0xc0000748
has-emergency: false
--------------------------------------------------------------------------------------------
$ 
=== Available nr-cli Commands ===
  status                - UE state info
  deregister normal     - 5G deregistration (~2s clean teardown)
  deregister disable-5g - Disable 5G
  ps-release 1          - Release PDU session 1
  ps-establish default  - Establish new PDU session


# 5. Monitoring & Metrics

## 5.1 — NF-specific Log Viewer

In [29]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
NF="amfd"     # amfd / smfd / upfd / ausfd / udmd / pcfd / nrfd / nwdafd
LINES=50

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== open5gs-'$NF' last '$LINES' lines ==='
    sudo journalctl -u open5gs-$NF -n $LINES --no-pager --output=short-iso 2>/dev/null | \
        awk '{
            if (/ERROR|error/)    print \"\033[31m\" \$0 \"\033[0m\"
            else if (/WARN|warn/) print \"\033[33m\" \$0 \"\033[0m\"
            else                  print \$0
        }'
"

=== open5gs-amfd last 50 lines ===
2026-04-25T21:50:32+0000 open5gs-ai-lab open5gs-amfd[402199]: 04/25 21:50:32.091: [sbi] INFO: [0e73715e-3383-41f1-99de-cf8cb50919b9] (NRF-profile-get) NF registered (../lib/sbi/nf-sm.c:81)
2026-04-25T21:50:32+0000 open5gs-ai-lab open5gs-amfd[402199]: 04/25 21:50:32.091: [sbi] INFO: Setup NF EndPoint(addr) [127.0.0.12:80] (../lib/sbi/context.c:2374)
2026-04-25T21:50:32+0000 open5gs-ai-lab open5gs-amfd[402199]: 04/25 21:50:32.091: [sbi] INFO: Setup NF EndPoint(addr) [127.0.0.12:7777] (../lib/sbi/context.c:2113)
2026-04-25T21:50:32+0000 open5gs-ai-lab open5gs-amfd[402199]: 04/25 21:50:32.091: [sbi] INFO: Setup NF EndPoint(addr) [127.0.0.12:7777] (../lib/sbi/context.c:2113)
2026-04-25T21:50:32+0000 open5gs-ai-lab open5gs-amfd[402199]: 04/25 21:50:32.091: [sbi] INFO: Setup NF EndPoint(addr) [127.0.0.12:7777] (../lib/sbi/context.c:2113)
2026-04-25T21:50:32+0000 open5gs-ai-lab open5gs-amfd[402199]: 04/25 21:50:32.092: [sbi] INFO: [0e7bd83a-3383-41f1-b907-4d7

## 5.2 — Live Interface Throughput (KB/s)

⚠️ Key constraint: uesimtun0 RX always shows 0 because gtp5g bypasses user-space capture. Use ogstun and ens4 counters for real measurements.

In [30]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
DURATION=30
INTERVAL=2

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo 'timestamp_ms,ue_tx,ue_rx,upf_tx,upf_rx'
read UE_RX1 UE_TX1 <<<\$(grep 'uesimtun0:' /proc/net/dev 2>/dev/null | \
    awk '{gsub(/:/,\" \"); print \$2, \$10}')
read OG_RX1 OG_TX1 <<<\$(grep 'ogstun:' /proc/net/dev 2>/dev/null | \
    awk '{gsub(/:/,\" \"); print \$2, \$10}')
T1=\$(date +%s%3N)
END=\$(( \$(date +%s) + $DURATION ))
sleep $INTERVAL
while [ \$(date +%s) -lt \$END ]; do
    T2=\$(date +%s%3N)
    read UE_RX2 UE_TX2 <<<\$(grep 'uesimtun0:' /proc/net/dev 2>/dev/null | \
        awk '{gsub(/:/,\" \"); print \$2, \$10}')
    read OG_RX2 OG_TX2 <<<\$(grep 'ogstun:' /proc/net/dev 2>/dev/null | \
        awk '{gsub(/:/,\" \"); print \$2, \$10}')
    python3 -c \"
b = [(\${UE_TX2:-0}-\${UE_TX1:-0}), (\${UE_RX2:-0}-\${UE_RX1:-0}),
     (\${OG_TX2:-0}-\${OG_TX1:-0}), (\${OG_RX2:-0}-\${OG_RX1:-0})]
dt = $INTERVAL
kbps = [round(max(0,x)/dt/1024,2) for x in b]
print(f'\${T2},{kbps[0]},{kbps[1]},{kbps[2]},{kbps[3]}')
\"
    UE_RX1=\$UE_RX2; UE_TX1=\$UE_TX2
    OG_RX1=\$OG_RX2; OG_TX1=\$OG_TX2
    T1=\$T2
    sleep $INTERVAL
done
" > /tmp/throughput_live.csv

# Plot in Colab
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('/tmp/throughput_live.csv',
                 names=['ts','ue_tx','ue_rx','upf_tx','upf_rx'])
df['time'] = pd.to_datetime(df['ts'], unit='ms')
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(df['time'], df['ue_tx'], label='UE TX', color='#2196F3')
axes[0].plot(df['time'], df['ue_rx'], label='UE RX (always 0 — gtp5g bypass)', color='#4CAF50', linestyle='--')
axes[0].set_title('UE Throughput (KB/s)'); axes[0].legend()
axes[1].plot(df['time'], df['upf_tx'], label='UPF TX (ogstun)', color='#FF9800')
axes[1].plot(df['time'], df['upf_rx'], label='UPF RX (ogstun)', color='#9C27B0')
axes[1].set_title('UPF Throughput (KB/s) — use this for real measurements'); axes[1].legend()
plt.tight_layout(); plt.savefig('throughput_live.png', dpi=150)
plt.show()

## 5.3 — UPF Prometheus Metrics DataFrame

In [31]:
import pandas as pd

result = ssh_run("curl -s http://127.0.0.7:9090/metrics")
lines = [l for l in result.stdout.split('\n') if l and not l.startswith('#')]

records = []
for line in lines:
    parts = line.rsplit(' ', 1)
    if len(parts) == 2:
        records.append({'metric': parts[0], 'value': parts[1]})

df_metrics = pd.DataFrame(records)
df_metrics['value'] = pd.to_numeric(df_metrics['value'], errors='coerce')

print("=== UPF Prometheus Metrics ===")
print(df_metrics.to_string(index=False))

# HELP fivegs_ep_n3_gtp_indatapktn3upf Number of incoming GTP data packets on the N3 interface
# TYPE fivegs_ep_n3_gtp_indatapktn3upf counter
fivegs_ep_n3_gtp_indatapktn3upf 0

# HELP fivegs_ep_n3_gtp_outdatapktn3upf Number of outgoing GTP data packets on the N3 interface
# TYPE fivegs_ep_n3_gtp_outdatapktn3upf counter
fivegs_ep_n3_gtp_outdatapktn3upf 0

# HELP fivegs_upffunction_sm_n4sessionestabreq Number of requested N4 session establishments
# TYPE fivegs_upffunction_sm_n4sessionestabreq counter
fivegs_upffunction_sm_n4sessionestabreq 1

# HELP fivegs_upffunction_sm_n4sessionreport Number of requested N4 session reports
# TYPE fivegs_upffunction_sm_n4sessionreport counter
fivegs_upffunction_sm_n4sessionreport 0

# HELP fivegs_upffunction_sm_n4sessionreportsucc Number of successful N4 session reports
# TYPE fivegs_upffunction_sm_n4sessionreportsucc counter
fivegs_upffunction_sm_n4sessionreportsucc 0

# HELP fivegs_upffunction_upf_sessionnbr Active Sessions
# TYPE fivegs_upffunction_

## 5.4 — Per-NF Resource Usage

In [32]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== Open5GS NF Process Resources ==='
    printf '%-20s %-8s %-8s %-8s\n' 'PROCESS' 'PID' 'CPU%' 'MEM%'
    for svc in amfd smfd upfd ausfd udmd pcfd nrfd nwdafd; do
        PID=\$(pgrep -f open5gs-\$svc | head -1)
        if [ -n \"\$PID\" ]; then
            LINE=\$(ps -p \$PID -o pid,pcpu,pmem --no-headers 2>/dev/null)
            CPU=\$(echo \$LINE | awk '{print \$2}')
            MEM=\$(echo \$LINE | awk '{print \$3}')
            printf '%-20s %-8s %-8s %-8s\n' \"open5gs-\$svc\" \"\$PID\" \"\$CPU\" \"\$MEM\"
        fi
    done

    echo ''
    echo '=== Disk Usage ==='
    df -h | grep -v tmpfs
    echo ''
    echo '=== Log File Sizes ==='
    sudo du -sh /var/log/open5gs/ 2>/dev/null
"

=== Open5GS NF Process Resources ===
PROCESS              PID      CPU%     MEM%    
open5gs-amfd         402199   0.0      0.4     
open5gs-smfd         402205   0.2      0.6     
open5gs-upfd         402209   0.0      0.4     
open5gs-ausfd        191229   0.0      0.4     
open5gs-udmd         191238   0.0      0.4     
open5gs-pcfd         191233   0.0      0.4     
open5gs-nrfd         191231   0.0      0.6     
open5gs-nwdafd       401881   0.1      0.1     

=== Disk Usage ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        29G   11G   19G  36% /
efivarfs        256K   18K  234K   8% /sys/firmware/efi/efivars
/dev/sda15      105M  6.1M   99M   6% /boot/efi

=== Log File Sizes ===
5.5M	/var/log/open5gs/


# 6. Subscriber Management

## 6.1 — List Subscribers

In [33]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== Subscriber List ==='
    mongosh --quiet open5gs --eval '
        db.subscribers.find({}, {
            imsi: 1, _id: 0,
            \"security.k\": 1, \"security.opc\": 1,
            \"slice.session.name\": 1
        }).forEach(function(d) {
            print(\"IMSI: \" + d.imsi + \" | Slice: \" + JSON.stringify(d.slice));
        });
    '
    echo ''
    echo 'Total subscribers:'
    mongosh --quiet open5gs --eval 'db.subscribers.countDocuments()'
"

=== Subscriber List ===
IMSI: 901700000000001 | Slice: [{"session":[{"name":"internet"}]}]
IMSI: 001010000000001 | Slice: [{"session":[{"name":"internet"}]}]
IMSI: 999700000000001 | Slice: [{"session":[{"name":"internet"}]}]
IMSI: 999700000000002 | Slice: [{"session":[{"name":"internet"}]}]
IMSI: 999700000000003 | Slice: [{"session":[{"name":"internet"}]}]

Total subscribers:
5


## 6.1b — Remove Orphan Subscribers (Wrong PLMN)

In [ ]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '=== Removing non-999/70 PLMN subscribers ==='
mongosh --quiet open5gs --eval '
  var result = db.subscribers.deleteMany({
    imsi: { \$not: /^999700/ }
  });
  print(\"Deleted \" + result.deletedCount + \" orphan subscriber(s)\");
  db.subscribers.find({},{imsi:1,_id:0}).forEach(d => print(\"  Remaining: \" + d.imsi));
'
"

## Cell 6.2 — Add Subscriber

In [34]:
'''

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
IMSI="999700000000002"
K="465B5CE8B199B49FAA5F0A2EE238A6BC"
OPC="E8ED289DEBA952E4283B54E88E6183CA"
DNN="internet"
DL_AMBR_MBPS=100
UL_AMBR_MBPS=50

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
mongosh --quiet open5gs --eval '
var imsi = \"$IMSI\";
var exists = db.subscribers.countDocuments({imsi: imsi});
if (exists > 0) { print(\"WARNING: IMSI \" + imsi + \" exists, updating...\"); }
var doc = {
    imsi: imsi, msisdn: [], imeisv: [],
    security: { k: \"$K\", op: null, opc: \"$OPC\", amf: \"8000\" },
    ambr: {
        downlink: { value: $DL_AMBR_MBPS, unit: 1 },
        uplink:   { value: $UL_AMBR_MBPS, unit: 1 }
    },
    slice: [{ sst: 1, default_indicator: true, session: [{
        name: \"$DNN\", type: 3,
        qos: { index: 9, arp: { priority_level: 8,
               pre_emption_capability: 1, pre_emption_vulnerability: 1 } },
        ambr: {
            downlink: { value: $DL_AMBR_MBPS, unit: 1 },
            uplink:   { value: $UL_AMBR_MBPS, unit: 1 }
        },
        ue: { addr: \"\" }, pcc_rule: []
    }] }],
    access_restriction_data: 32, subscriber_status: 0,
    network_access_mode: 0, subscribed_rau_tau_timer: 12, __v: 0
};
db.subscribers.replaceOne({imsi: imsi}, doc, {upsert: true});
print(\"OK: IMSI \" + imsi + \" saved.\");
'
"


'''

'\n\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\nIMSI="999700000000002"\nK="465B5CE8B199B49FAA5F0A2EE238A6BC"\nOPC="E8ED289DEBA952E4283B54E88E6183CA"\nDNN="internet"\nDL_AMBR_MBPS=100\nUL_AMBR_MBPS=50\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\nmongosh --quiet open5gs --eval \'\nvar imsi = "$IMSI";\nvar exists = db.subscribers.countDocuments({imsi: imsi});\nif (exists > 0) { print("WARNING: IMSI " + imsi + " exists, updating..."); }\nvar doc = {\n    imsi: imsi, msisdn: [], imeisv: [],\n    security: { k: "$K", op: null, opc: "$OPC", amf: "8000" },\n    ambr: {\n        downlink: { value: $DL_AMBR_MBPS, unit: 1 },\n        uplink:   { value: $UL_AMBR_MBPS, unit: 1 }\n    },\n    slice: [{ sst: 1, default_indicator: true, session: [{\n        name: "$DNN", type: 3,\n        qos: { index: 9, arp: { priority_level: 8,\n               pre_emption_capability: 1, pre_emption_vulnerability: 1 } },\n        ambr: {\n

## 6.3 — Delete Subscriber

In [35]:
'''

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
IMSI_TO_DELETE="999700000000002"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo 'Deleting IMSI: $IMSI_TO_DELETE'
    mongosh --quiet open5gs --eval '
        var r = db.subscribers.deleteOne({imsi: \"$IMSI_TO_DELETE\"});
        print(\"Deleted: \" + r.deletedCount + \" document(s)\");
    '
"

'''

'\n\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\nIMSI_TO_DELETE="999700000000002"\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    echo \'Deleting IMSI: $IMSI_TO_DELETE\'\n    mongosh --quiet open5gs --eval \'\n        var r = db.subscribers.deleteOne({imsi: "$IMSI_TO_DELETE"});\n        print("Deleted: " + r.deletedCount + " document(s)");\n    \'\n"\n\n'

# 7. Configuration Management

## 7.1 — View NF Config

In [36]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
NF="nwdaf"   # amf / smf / upf / ausf / udm / pcf / nrf

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== /etc/open5gs/$NF.yaml ==='
    cat /etc/open5gs/$NF.yaml
"

=== /etc/open5gs/nwdaf.yaml ===
nwdaf:
  nf_instance_id: ""
  plmn_mcc: "999"
  plmn_mnc: "70"
  sbi_bind_address: "127.0.0.1"
  sbi_port: 7779
  nf_service_names:
    AMF:  "amfd"
    SMF:  "smfd"
    UPF:  "upfd"
    AUSF: "ausfd"
    UDM:  "udmd"
    PCF:  "pcfd"
    NRF:  "nrfd"
    UDR:  "udrd"
    BSF:  "bsfd"
    NSSF: "nssfd"
  throughput_interfaces:
    - "ogstun"
    - "ogstun2"
    - "ogstun3"
    - "ens4"
  throughput_history_size: 360
  collection_interval_seconds: 10
  amf_journal_lines: 500
  smf_journal_lines: 500
  supi_regex: "imsi-(\\d{15})"
  mongodb_uri: "mongodb://127.0.0.1:27017"
  mongodb_db:  "open5gs"
  nrf_uri: "http://127.0.0.1:7777"
  nrf_register_on_startup: true
  nrf_heartbeat_interval_seconds: 60
  model_dir: "/opt/nwdaf/models"
  anomaly_contamination: 0.10
  anomaly_min_samples: 10
  baseline_stddev_min_kbps: 0.5
  ewma_alpha: 0.3
  log_level: "info"
  log_file: "/var/log/open5gs/nwdaf.log"
  history_backend: "sqlite"
  history_db_path: "/opt/nwdaf/hi

## 7.2 — Edit NF Config in Python

In [37]:
import yaml, subprocess

NF = "amf"
LOCAL_CFG = f"/tmp/{NF}_edit.yaml"

# 1) Pull config to Colab
scp_get(f"/etc/open5gs/{NF}.yaml", LOCAL_CFG)

# 2) Edit in Python
with open(LOCAL_CFG) as f:
    cfg = yaml.safe_load(f)

# --- Make your changes here ---
# Example: update integrity algorithms
# cfg['amf']['security']['integrity_order'] = ['NIA2', 'NIA1']
# cfg['amf']['security']['ciphering_order']  = ['NEA2', 'NEA1', 'NEA0']
# ------------------------------

print("Current security config:")
print(yaml.dump(cfg.get('amf', {}).get('security', {}), default_flow_style=False))

# 3) Push back and restart (uncomment when ready)
# with open(LOCAL_CFG, 'w') as f:
#     yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
# scp_put(LOCAL_CFG, f"/etc/open5gs/{NF}.yaml")
# ssh_run(f"systemctl restart open5gs-{NF}d", sudo=True)
# print(f"✅ {NF} config updated and service restarted.")

✅ Downloaded: /tmp/amf_edit.yaml/amf.yaml
Current security config:
ciphering_order:
- NEA0
- NEA1
- NEA2
integrity_order:
- NIA2
- NIA1
- NIA0



## 7.3 — View UERANSIM Config

In [38]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
TARGET="gnb"   # gnb or ue

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== /opt/UERANSIM/config/open5gs-$TARGET.yaml ==='
    cat /opt/UERANSIM/config/open5gs-$TARGET.yaml
"

=== /opt/UERANSIM/config/open5gs-gnb.yaml ===
mcc: '999'          # Mobile Country Code value
mnc: '70'           # Mobile Network Code value (2 or 3 digits)

nci: '0x000000010'  # NR Cell Identity (36-bit)
idLength: 32        # NR gNB ID length in bits [22...32]
tac: 1              # Tracking Area Code

linkIp: 127.0.0.1   # gNB's local IP address for Radio Link Simulation (Usually same with local IP)
ngapIp: 127.0.0.1   # gNB's local IP address for N2 Interface (Usually same with local IP)
gtpIp: 127.0.0.1    # gNB's local IP address for N3 Interface (Usually same with local IP)

# List of AMF address information
amfConfigs:
  - address: 127.0.0.5
    port: 38412

# List of supported S-NSSAIs by this gNB
slices:
  - sst: 1

# Indicates whether or not SCTP stream number errors should be ignored.
ignoreStreamIds: true


# 8. NWDAF Operations

## 8.1 — NWDAF Health Check

In [39]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '════════════════════════════════════════════════════'
echo '  NWDAF STATUS — '\$(date '+%Y-%m-%d %H:%M:%S')
echo '════════════════════════════════════════════════════'

echo '[1] Service'
systemctl is-active open5gs-nwdafd | xargs -I{} echo '  Status: {}'

echo ''
echo '[2] SBI Endpoint (TS 29.520 health)'
curl -s http://localhost:7779/nwdaf-analytics/v1/health | \
    python3 -c '
import sys, json
d = json.load(sys.stdin)
print(\"  Status    :\", d.get(\"status\", \"?\"))
print(\"  NF Type   :\", d.get(\"nfType\", \"?\"))
iid = d.get(\"nfInstanceId\",\"?\")
print(\"  Instance  :\", iid[:8]+\"...\" if len(iid) > 8 else iid)
nf = d.get(\"nfProfile\", {})
ids = nf.get(\"nwdafInfo\", {}).get(\"analyticsIds\", [])
if ids:
    print(\"  Analytics :\", len(ids), \"IDs supported\")
'

echo ''
echo '[3] ML Models'
ls -lh /opt/nwdaf/models/ 2>/dev/null || echo '  No models yet — run Cell 9.1'

echo ''
echo '[4] 3GPP Compliance'
echo '  ✅ TS 23.288 Table 2.1-1 — All 7 Analytics IDs'
echo '  ✅ TS 29.520             — REST SBI'
echo '  ✅ TS 29.510             — NRF registration payload'
echo '  ✅ TS 28.554             — KPI data collection'
echo '  ⚠️  TS 29.520 §5.3.3    — Subscription push not implemented'
NRF_HB=\$(grep 'nrf_heartbeat_interval_seconds' /etc/open5gs/nwdaf.yaml 2>/dev/null | awk '{print \$2}')
if [ \"\${NRF_HB:-0}\" -gt 0 ] 2>/dev/null; then
    echo \"  ✅ TS 29.510 §5.3.2.4  — NRF heartbeat every \${NRF_HB}s (configured)\"
else
    echo '  ⚠️  TS 29.510 §5.3.2.4  — No NRF heartbeat'
fi
echo '════════════════════════════════════════════════════'
"

════════════════════════════════════════════════════
  NWDAF STATUS — 2026-04-25 21:52:44
════════════════════════════════════════════════════
[1] Service
  Status: active

[2] SBI Endpoint (TS 29.520 health)
  Status    : UP

[3] ML Models
total 28K
-rw-r--r-- 1 root root 28K Apr 25 21:44 isolation_forest.json

[4] 3GPP Compliance
  ✅ TS 23.288 Table 2.1-1 — All 7 Analytics IDs
  ✅ TS 29.520             — REST SBI
  ✅ TS 29.510             — NRF registration payload
  ✅ TS 28.554             — KPI data collection
  ⚠️  TS 29.520 §5.3.3    — Subscription push not implemented
  ⚠️  TS 29.510 §5.3.2.4  — No NRF heartbeat
════════════════════════════════════════════════════


Traceback (most recent call last):
  File "<string>", line 4, in <module>
KeyError: 'nfProfile'


## 8.2 — Query a Specific Analytics ID

In [40]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
ANALYTICS_ID="NF_LOAD"
# Options: NF_LOAD | UE_MOBILITY | UE_COMMUNICATION | ABNORMAL_BEHAVIOUR
#          QoS_SUSTAINABILITY | SERVICE_EXPERIENCE | NETWORK_PERFORMANCE

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
curl -s 'http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=$ANALYTICS_ID' | \
    python3 -m json.tool
"

{
    "analData": {
        "analyticsId": "NF_LOAD",
        "confidence": 95,
        "nfLoadLevelList": [
            {
                "nfLoadLevelInfo": {
                    "nfCpuUsage": 0.05,
                    "nfLoadLevel": 0.0,
                    "nfLoadLevelLabel": "LOW",
                    "nfMemoryUsage": 34336
                },
                "nfStatus": "active",
                "nfType": "AMF"
            },
            {
                "nfLoadLevelInfo": {
                    "nfCpuUsage": 78.08,
                    "nfLoadLevel": 0.0,
                    "nfLoadLevelLabel": "LOW",
                    "nfMemoryUsage": 34292
                },
                "nfStatus": "active",
                "nfType": "AUSF"
            },
            {
                "nfLoadLevelInfo": {
                    "nfCpuUsage": 77.76,
                    "nfLoadLevel": 0.0,
                    "nfLoadLevelLabel": "LOW",
                    "nfMemoryUsage": 34172
                }

## 8.3 — Poll All Analytics IDs

In [41]:
import subprocess, json, time, pandas as pd
from IPython.display import display, clear_output

def nwdaf_query(aid):
    cmd = ["gcloud","compute","ssh", VM_NAME,
           f"--project={PROJECT_ID}", f"--zone={ZONE}",
           f"--command=curl -s 'http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId={aid}'"]
    return json.loads(subprocess.check_output(cmd, timeout=30).decode())

POLL_INTERVAL = 5   # seconds between polls
POLL_COUNT    = 6   # number of iterations

records = []
for i in range(POLL_COUNT):
    ts  = time.strftime('%H:%M:%S')
    net = nwdaf_query("NETWORK_PERFORMANCE").get('analData', {})
    qos = nwdaf_query("QoS_SUSTAINABILITY").get('analData', {})
    abn = nwdaf_query("ABNORMAL_BEHAVIOUR").get('analData', {})
    nfl = nwdaf_query("NF_LOAD").get('analData', {})
    svc = nwdaf_query("SERVICE_EXPERIENCE").get('analData', {})

    records.append({
        'Time'      : ts,
        'Net Grade' : net.get('scoreLabel', '?'),
        'Net Score' : round(net.get('overallScore', 0), 2),
        'DL (Kbps)' : round(qos.get('currentDlKbps', 0), 2),
        'UL (Kbps)' : round(qos.get('currentUlKbps', 0), 2),
        'DL Trend'  : qos.get('dlTrend', '?'),
        'QoS Risk'  : qos.get('violationRisk', '?'),
        'MOS'       : f"{svc.get('mosScore','?')} ({svc.get('mosCategory','?')})",
        'Anomaly'   : '\u26a0\ufe0f YES' if abn.get('anomalyDetected') else '\u2705 NO',
        'Anomaly%'  : round(abn.get('anomalyPct', 0), 1),
        'NF Health' : nfl.get('recommendation', '?'),
    })

    clear_output(wait=True)
    display(pd.DataFrame(records))
    print(f"Poll {i+1}/{POLL_COUNT} — next in {POLL_INTERVAL}s...")
    if i < POLL_COUNT - 1:
        time.sleep(POLL_INTERVAL)

print("\n✅ Monitoring complete")

,Time,Net Grade,Net Score,DL (Kbps),UL (Kbps),QoS,Anomaly,Anomaly%,NF Health
0,21:52:52,?,60.144599,9.764282,8.271244,?,⚠️ YES,100.0,?
1,21:53:27,?,60.099126,13.647449,7.870359,?,⚠️ YES,100.0,?
2,21:53:58,?,60.123307,8.387906,6.520165,?,⚠️ YES,100.0,?
3,21:54:32,?,60.525063,11.229685,6.788078,?,⚠️ YES,100.0,?
4,21:55:04,?,60.132095,7.446608,6.103609,?,⚠️ YES,100.0,?
5,21:55:38,?,60.046607,10.088095,5.921926,?,⚠️ YES,100.0,?


Poll 6/6 — next in 5s...

✅ Monitoring complete


## 8.4 — Subscription Lifecycle (TS 29.520)

In [42]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
# POST — create subscription
echo '=== POST /subscriptions ==='
SUB_RESP=\$(curl -s -X POST http://localhost:7779/nwdaf-analytics/v1/subscriptions \
    -H 'Content-Type: application/json' \
    -d '{\"analyticsId\": \"NF_LOAD\", \"notifUri\": \"http://localhost:9999/notify\", \"repPeriod\": 30}')
echo \$SUB_RESP | python3 -m json.tool
SUB_ID=\$(echo \$SUB_RESP | python3 -c 'import sys,json; print(json.load(sys.stdin).get(\"subId\",\"\"))' 2>/dev/null)

echo ''
# GET — retrieve subscription
echo '=== GET /subscriptions/{subId} ==='
[ -n \"\$SUB_ID\" ] && \
    curl -s http://localhost:7779/nwdaf-analytics/v1/subscriptions/\$SUB_ID | \
    python3 -m json.tool || echo 'No subId captured'

echo ''
# DELETE — remove subscription
echo '=== DELETE /subscriptions/{subId} ==='
[ -n \"\$SUB_ID\" ] && \
    curl -s -X DELETE http://localhost:7779/nwdaf-analytics/v1/subscriptions/\$SUB_ID && \
    echo '  Subscription deleted' || echo 'No subId to delete'
"

=== POST /subscriptions ===
{
    "analyticsId": "NF_LOAD",
    "createdAt": "2026-04-25T21:56:22Z",
    "notifUri": "http://localhost:9999/notify",
    "status": "ACTIVE",
    "subId": "sub-cf9e06f4eec988de"
}

=== GET /subscriptions/{subId} ===
{
    "analyticsId": "NF_LOAD",
    "createdAt": "2026-04-25T21:56:22Z",
    "notifUri": "http://localhost:9999/notify",
    "status": "ACTIVE",
    "subId": "sub-cf9e06f4eec988de"
}

=== DELETE /subscriptions/{subId} ===
  Subscription deleted


# 9. ML Model Lifecycle

⚠️ Critical: Run Cell 9.1 while UE sessions are active and generating traffic. Training on idle data makes the Isolation Forest flag normal traffic as anomalous (BASELINE_TOO_LOW guard will prevent worst-case false positives, but model quality depends on real traffic patterns).

## 9.1 — Collect Data + Train + Deploy Anomaly Model

## 9.0 — ML Training Readiness Check

In [ ]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
NWDAF_STATUS_JSON=\$(curl -s --connect-timeout 5 \
  'http://localhost:7779/nwdaf-analytics/v1/status' 2>/dev/null)

echo 'Data collection status:'
echo \$NWDAF_STATUS_JSON | python3 -m json.tool 2>/dev/null || \
  echo '  (status endpoint not available — check C++ NWDAF version)'

echo ''
echo 'Current model:'
ls -lh /opt/nwdaf/models/ 2>/dev/null || echo '  No model trained yet'
echo ''
echo 'NWDAF uptime (data collection time):'
systemctl status open5gs-nwdafd | grep 'Active:' | head -1
echo ''
echo 'Recommendation:'
echo '  Run scenarios in Section 10 for at least 20 minutes before Cell 9.1'
echo '  Target: ≥120 data points at 10s collection interval'
# ⚠️ Requires: ≥120 data points (~20 min at 10s interval)
# Check current count: curl http://localhost:7779/nwdaf-analytics/v1/status
"

In [43]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '=== Triggering Native C++ ML Model Training ==='
echo 'Training Isolation Forest...'
curl -s -X POST http://localhost:7779/nwdaf-analytics/v1/train | python3 -m json.tool
"


=== Triggering Native C++ ML Model Training ===
Training Isolation Forest...
{
    "dataPoints": 39,
    "nFeatures": 5,
    "status": "trained",
    "ts": "2026-04-25T21:56:34Z"
}


## 9.2 — Check Model Status

In [44]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '=== ML Models in /opt/nwdaf/models/ ==='
ls -lh /opt/nwdaf/models/ 2>/dev/null || echo 'No models found'

echo ''
echo '=== Anomaly model test via NWDAF API ==='
RAW=\$(curl -s --connect-timeout 5 \
    'http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=ABNORMAL_BEHAVIOUR' \
    2>/dev/null)

if [ -z \"\$RAW\" ]; then
    echo '  ❌ Empty response — NWDAF may not be running or port 7779 unreachable'
    echo '     Check: systemctl is-active open5gs-nwdafd'
else
    echo \"\$RAW\" | python3 -c '
import sys, json
raw = sys.stdin.read().strip()
try:
    d = json.loads(raw)
    status = d.get(\"status\") or d.get(\"analData\", {}).get(\"status\", \"N/A\")
    print(\"  Raw response:\")
    print(json.dumps(d, indent=2))
    if status == \"BASELINE_TOO_LOW\":
        print()
        print(\"  ⚠️  BASELINE_TOO_LOW — model was trained on idle/zero-traffic data.\")
        print(\"     Fix: start UE sessions, generate traffic, then re-run Cell 9.1.\")
    elif d.get(\"analData\", {}).get(\"anomalyDetected\") is not None:
        abn = d[\"analData\"][\"anomalyDetected\"]
        pct = d[\"analData\"].get(\"anomalyPct\", 0)
        print()
        print(\"  ✅ Model operational\")
        print(f\"     Anomaly detected: {abn}  ({pct:.1f}%)\")
except json.JSONDecodeError:
    print(\"  ❌ Invalid JSON:\", repr(raw[:200]))
' 2>&1
fi

echo ''
echo '=== NWDAF Service Status ==='
systemctl is-active open5gs-nwdafd 2>/dev/null || echo 'not-installed'
" || true
# ↑ trailing '|| true' prevents %%bash CalledProcessError on any non-zero remote exit

=== ML Models in /opt/nwdaf/models/ ===
total 32K
-rw-r--r-- 1 root root 32K Apr 25 21:56 isolation_forest.json

=== Anomaly model test via NWDAF API ===
  Raw response:
{
  "analData": {
    "analyticsId": "ABNORMAL_BEHAVIOUR",
    "anomalyDetected": true,
    "anomalyIndices": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      25,
      26,
      29,
      30,
      31,
      32,
      33,
      34,
      36,
      37,
      38
    ],
    "anomalyPct": 84.61538461538461,
    "anomalyType": "UNEXPECTED_WAKEUP",
    "avgAnomalyScore": -0.643255602491071,
    "baselineDlStd": 8.415845407337958,
    "confidence": 10,
    "dataPoints": 39,
    "ts": "2026-04-25T21:56:41Z"
  },
  "analyticsId": "ABNORMAL_BEHAVIOUR",
  "confidence": 10,
  "requestTime": "2026-04-25T21:56:41Z",
  "timeStampGen": "2026-04-25T21:56:41Z",
  "vali

# 10. Traffic Scenarios

## 10.1 — Scenario A: General Traffic Profile

In [45]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

cat > /tmp/scenario_a.sh << 'SCRIPT'
#!/bin/bash
LOG="/tmp/scenario_a.csv"
NR="/opt/UERANSIM/build"
UE="uesimtun0"
echo "timestamp_ms,scenario,ue_ip,target,protocol,bytes,rtt_ms,status" > $LOG
log() { echo "$(date +%s%3N),$1,10.45.0.3,$2,$3,$4,$5,$6" >> $LOG; }

echo "[A] ICMP — 20 pings"
for i in $(seq 1 20); do
    cd $NR
    RESULT=$(./nr-binder $UE ping -c 1 -q 8.8.8.8 2>&1)
    RTT=$(echo "$RESULT" | grep rtt | awk -F'/' '{print $5}')
    LOSS=$(echo "$RESULT" | grep transmitted | awk '{print $6}' | tr -d '%')
    STATUS=$([ "${LOSS:-100}" -eq 0 ] && echo "OK" || echo "LOSS")
    log "A_ICMP" "8.8.8.8" "ICMP" "64" "${RTT:-0}" "$STATUS"
done

echo "[A] DNS — 10 lookups"
for host in google.com github.com cloudflare.com youtube.com facebook.com \
            twitter.com amazon.com netflix.com reddit.com wikipedia.org; do
    cd $NR
    T1=$(date +%s%3N)
    ./nr-binder $UE nslookup $host 8.8.8.8 > /dev/null 2>&1
    T2=$(date +%s%3N); RTT=$((T2-T1))
    log "A_DNS" "$host" "DNS" "128" "$RTT" "OK"
done

echo "CSV: $LOG ($(wc -l < $LOG) records)"
SCRIPT

gcloud compute scp /tmp/scenario_a.sh \
    $VM_NAME:/tmp/scenario_a.sh --project=$PROJECT_ID --zone=$ZONE

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE \
    --command="sudo bash /tmp/scenario_a.sh"

gcloud compute scp $VM_NAME:/tmp/scenario_a.csv \
    ./scenario_a.csv --project=$PROJECT_ID --zone=$ZONE

[A] ICMP — 20 pings
[A] DNS — 10 lookups
CSV: /tmp/scenario_a.csv (31 records)


## 10.2 — Scenario C: Throughput Measurement

This cell uses the fixed method: synchronous traffic-then-measure loops with ogstun counters (not uesimtun0 which never updates RX due to gtp5g bypass).

In [46]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
set +e
LOG='/tmp/scenario_c.csv'
NR='/opt/UERANSIM/build'
CFG_GNB='/opt/UERANSIM/config/open5gs-gnb.yaml'
CFG_UE='/opt/UERANSIM/config/open5gs-ue.yaml'

# ── Step 1: Ensure gNB is running ────────────────────────────────────────────
if ! pgrep -x nr-gnb > /dev/null 2>&1; then
    echo '⚙️  gNB not running — starting...'
    sudo nohup \$NR/nr-gnb -c \$CFG_GNB > /root/gnb.log 2>&1 &
    sleep 4
fi
pgrep -x nr-gnb > /dev/null && echo '✅ gNB: running' || echo '❌ gNB: failed to start'

# ── Step 2: Ensure UE is running and tunnel is up ─────────────────────────────
UE_IFACE=\$(ip a | grep -o 'uesimtun[0-9]*' | head -1)

if [ -z \"\$UE_IFACE\" ]; then
    echo '⚙️  No uesimtun interface — starting UE...'
    sudo nohup \$NR/nr-ue -c \$CFG_UE > /root/ue.log 2>&1 &
    echo '   Waiting for PDU session (up to 20s)...'
    for i in \$(seq 1 40); do
        sleep 0.5
        UE_IFACE=\$(ip a | grep -o 'uesimtun[0-9]*' | head -1)
        [ -n \"\$UE_IFACE\" ] && echo \"✅ Tunnel up: \$UE_IFACE\" && break
    done
fi

if [ -z \"\$UE_IFACE\" ]; then
    echo '❌ Could not bring up uesimtun after 20s.'
    echo '   Check: sudo journalctl -u open5gs-amfd -n 30 --no-pager'
    echo '   Check: cat /root/ue.log | tail -20'
    exit 0
fi

echo \"Using interface: \$UE_IFACE\"

# ── Step 3: Connectivity pre-check ───────────────────────────────────────────
echo ''
echo '--- Connectivity pre-check ---'
cd \$NR && sudo ./nr-binder \$UE_IFACE ping -c 1 -q 8.8.8.8 > /dev/null 2>&1 \
    && echo '✅ nr-binder connectivity OK' \
    || echo '❌ nr-binder connectivity FAIL — check UE tunnel'

# ── Step 4: Write CSV header ──────────────────────────────────────────────────
echo 'timestamp_ms,phase,app_dl_kbps,app_ul_kbps' > \$LOG

# ── Phase 1: Idle baseline (genuine 0 KB/s, no traffic) ──────────────────────
echo ''
echo '--- Phase 1: Idle (5 samples × 2s, 0 KB/s by definition) ---'
for i in \$(seq 1 5); do
    TS=\$(date +%s%3N)
    echo \"\$TS,IDLE,0,0\" >> \$LOG
    echo \"  [\$i/5] IDLE  app_dl=0 app_ul=0 KB/s\"
    sleep 2
done

# ── Phase 2: Download via wget (application-layer byte count) ─────────────────
echo ''
echo '--- Phase 2: Download (wget 10MB via nr-binder) ---'
cd \$NR
T_START=\$(date +%s%3N)
WGET_OUT=\$(sudo ./nr-binder \$UE_IFACE \
    wget -O /dev/null --progress=dot:mega \
    http://speedtest.tele2.net/10MB.zip 2>&1)
T_END=\$(date +%s%3N)
BYTES_MB=\$(echo \"\$WGET_OUT\" | grep -oP '[0-9]+(?= MB)' | tail -1)
BYTES_MB=\${BYTES_MB:-0}
ELAPSED_S=\$(python3 -c \"print(max(0.1,(\$T_END-\$T_START)/1000.0))\")
DL_KBPS=\$(python3 -c \"print(round(\$BYTES_MB*1024*1024/\$ELAPSED_S/1024,2))\")
TS=\$(date +%s%3N)
echo \"\$TS,DOWNLOAD,\$DL_KBPS,0\" >> \$LOG
echo \"  DOWNLOAD  app_dl=\${DL_KBPS} KB/s  (bytes=\${BYTES_MB}MB elapsed=\${ELAPSED_S}s)\"

# ── Phase 3: Upload via ping flood (packet-count × packet-size / elapsed) ─────
echo ''
echo '--- Phase 3: Upload burst (ping flood 5000 × 1400-byte packets) ---'
cd \$NR
T_START=\$(date +%s%3N)
PING_OUT=\$(sudo ./nr-binder \$UE_IFACE ping -f -c 5000 -s 1400 -q 8.8.8.8 2>&1)
T_END=\$(date +%s%3N)
PACKETS_TX=\$(echo \"\$PING_OUT\" | grep -oP '[0-9]+(?= packets transmitted)' | head -1)
PACKETS_TX=\${PACKETS_TX:-0}
ELAPSED_S=\$(python3 -c \"print(max(0.1,(\$T_END-\$T_START)/1000.0))\")
UL_KBPS=\$(python3 -c \"print(round(\$PACKETS_TX*1400/\$ELAPSED_S/1024,2))\")
TS=\$(date +%s%3N)
echo \"\$TS,UPLOAD,0,\$UL_KBPS\" >> \$LOG
echo \"  UPLOAD  app_ul=\${UL_KBPS} KB/s  (packets=\${PACKETS_TX} elapsed=\${ELAPSED_S}s)\"

echo ''
echo \"✅ CSV ready: \$LOG  (\$(wc -l < \$LOG) records)\"
cat \$LOG
" 2>&1

# ── Download CSV to Colab ─────────────────────────────────────────────────────
gcloud compute scp $VM_NAME:/tmp/scenario_c.csv \
    ./scenario_c.csv \
    --project=$PROJECT_ID --zone=$ZONE 2>/dev/null \
    && echo "✅ Downloaded: scenario_c.csv" \
    || echo "⚠️  SCP failed — CSV may not exist if UE never came up"

✅ gNB: running
Using interface: uesimtun0

--- Phase 1: Idle (10s, 5 samples) ---
  [1/5] IDLE  ogstun_tx=0 ogstun_rx=0 KB/s
  [2/5] IDLE  ogstun_tx=0 ogstun_rx=0 KB/s
  [3/5] IDLE  ogstun_tx=0 ogstun_rx=0 KB/s
  [4/5] IDLE  ogstun_tx=0 ogstun_rx=0 KB/s
  [5/5] IDLE  ogstun_tx=0 ogstun_rx=0 KB/s

--- Phase 2: Download (wget 10MB, 8 samples ~16s) ---
  [1/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [2/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [3/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [4/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [5/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [6/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [7/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [8/8] DOWNLOAD  ogstun_tx=0 ogstun_rx=0 KB/s

--- Phase 3: Upload burst (ping flood, 5 samples ~10s) ---
  [1/5] UPLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [2/5] UPLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [3/5] UPLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [4/5] UPLOAD  ogstun_tx=0 ogstun_rx=0 KB/s
  [5/5] UPLOAD  ogstun_tx=0 ogst

In [47]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
set +e
echo '════════════════════════════════════════════'
echo '  UE SESSION FAILURE DIAGNOSIS'
echo '  '\$(date '+%Y-%m-%d %H:%M:%S')
echo '════════════════════════════════════════════'

echo ''
echo '[1] All NF service status'
for svc in nrfd amfd smfd upfd ausfd udmd pcfd; do
    printf '  %-8s: %s\n' \$svc \$(systemctl is-active open5gs-\$svc 2>/dev/null)
done

echo ''
echo '[2] UE log — last 40 lines'
cat /root/ue.log 2>/dev/null | tail -40 || echo '  /root/ue.log not found'

echo ''
echo '[3] AMF log — last 40 lines'
sudo journalctl -u open5gs-amfd -n 40 --no-pager 2>/dev/null

echo ''
echo '[4] SMF log — last 20 lines'
sudo journalctl -u open5gs-smfd -n 20 --no-pager 2>/dev/null

echo ''
echo '[5] Current tunnel interfaces'
ip a | grep -E 'uesimtun|ogstun' || echo '  none'

echo ''
echo '[6] UERANSIM config — PLMN / AMF address check'
grep -E 'mcc|mnc|amfConfigs|host|port' /opt/UERANSIM/config/open5gs-ue.yaml 2>/dev/null
echo '---'
grep -E 'mcc|mnc|ngapIpAddress|address' /opt/UERANSIM/config/open5gs-gnb.yaml 2>/dev/null

echo ''
echo '[7] Open5GS AMF config — PLMN / NGAP check'
grep -E 'mcc|mnc|addr|ngap' /etc/open5gs/amf.yaml 2>/dev/null | head -30

echo ''
echo '[8] UPF / PFCP association'
sudo journalctl -u open5gs-upfd -n 20 --no-pager 2>/dev/null | grep -iE 'pfcp|assoc|error'

echo ''
echo '[9] MongoDB subscriber count'
mongosh --quiet open5gs --eval 'db.subscribers.countDocuments()' 2>/dev/null
" 2>&1

════════════════════════════════════════════
  UE SESSION FAILURE DIAGNOSIS
  2026-04-25 22:00:56
════════════════════════════════════════════

[1] All NF service status
  nrfd    : active
  amfd    : active
  smfd    : active
  upfd    : active
  ausfd   : active
  udmd    : active
  pcfd    : active

[2] UE log — last 40 lines
UERANSIM v3.2.7
[2026-04-25 21:51:02.836] [nas] [info] UE switches to state [MM-DEREGISTERED/PLMN-SEARCH]
[2026-04-25 21:51:02.836] [rrc] [debug] New signal detected for cell[1], total [1] cells in coverage
[2026-04-25 21:51:02.837] [nas] [info] Selected plmn[999/70]
[2026-04-25 21:51:02.837] [rrc] [info] Selected cell plmn[999/70] tac[1] category[SUITABLE]
[2026-04-25 21:51:02.837] [nas] [info] UE switches to state [MM-DEREGISTERED/PS]
[2026-04-25 21:51:02.837] [nas] [info] UE switches to state [MM-DEREGISTERED/NORMAL-SERVICE]
[2026-04-25 21:51:02.837] [nas] [debug] Initial registration required due to [MM-DEREG-NORMAL-SERVICE]
[2026-04-25 21:51:02.837] [nas] 

# 11. Fault Management & Auto-Diagnosis

## 11.1 — Auto-Diagnosis (Run First When Something Is Wrong)

In [48]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
ISSUES=0
echo '════════════════════════════════════════'
echo '  OPEN5GS AUTO-DIAGNOSIS'
echo '  '\$(date '+%Y-%m-%d %H:%M:%S')
echo '════════════════════════════════════════'

echo ''
echo '[1] NF Service Status'
for svc in amfd smfd upfd ausfd udmd pcfd nrfd; do
    STATUS=\$(systemctl is-active open5gs-\$svc 2>/dev/null)
    if [ \"\$STATUS\" != 'active' ]; then
        echo \"  ❌ FAULT: open5gs-\$svc = \$STATUS\"
        ISSUES=\$(( ISSUES + 1 ))
    fi
done
[ \"\$ISSUES\" -eq 0 ] && echo '  ✅ All NFs active'

echo ''
echo '[2] PFCP Association (SMF ↔ UPF)'
PFCP_OK=\$(sudo journalctl -u open5gs-smfd -n 50 --no-pager 2>/dev/null | \
    grep -c 'PFCP.*associated\|pfcp.*association')
[ \"\$PFCP_OK\" -gt 0 ] && echo \"  ✅ PFCP OK (\$PFCP_OK events)\" || \
    echo '  ❌ PFCP: No recent association — check UPF/SMF'

echo ''
echo '[3] NAT Masquerade Rules'
MASQ=\$(sudo iptables -t nat -L POSTROUTING -n 2>/dev/null | grep -c MASQUERADE)
[ \"\$MASQ\" -gt 0 ] && echo '  ✅ MASQUERADE rule present' || \
    echo '  ❌ MASQUERADE rule MISSING — run NAT repair (Cell 12.2)'

echo ''
echo '[4] IP Forwarding'
FWD=\$(sysctl -n net.ipv4.ip_forward 2>/dev/null)
[ \"\$FWD\" = '1' ] && echo '  ✅ IP forwarding: enabled' || \
    echo '  ❌ IP forwarding DISABLED — run: sysctl -w net.ipv4.ip_forward=1'

echo ''
echo '[5] MongoDB'
MONGO_OK=\$(mongosh --quiet open5gs --eval 'db.runCommand({ping:1}).ok' 2>/dev/null)
[ \"\$MONGO_OK\" = '1' ] && echo '  ✅ MongoDB: reachable' || \
    echo '  ❌ MongoDB: unreachable'

echo ''
echo '[6] gtp5g Kernel Module'
lsmod | grep -q gtp5g && echo '  ✅ gtp5g loaded' || \
    echo '  ❌ gtp5g NOT loaded — UPF cannot forward packets'

echo ''
echo '[7] UE Tunnel Interfaces'
TUN_COUNT=\$(ip a | grep -c 'uesimtun')
[ \"\$TUN_COUNT\" -gt 0 ] && echo \"  ✅ \$TUN_COUNT tunnel interface(s) active\" || \
    echo '  ⚠️  No uesimtun interfaces — no active PDU sessions'

echo ''
echo '[8] UPF Prometheus Endpoint'
curl -s --connect-timeout 3 http://127.0.0.7:9090/metrics > /dev/null 2>&1 && \
    echo '  ✅ UPF metrics endpoint reachable' || \
    echo '  ❌ UPF metrics endpoint unreachable'

echo ''
echo '[9] NWDAF Service'
NWDAF_STATUS=\$(systemctl is-active open5gs-nwdafd 2>/dev/null || echo 'not-installed')
[ \"\$NWDAF_STATUS\" = 'active' ] && echo '  ✅ NWDAF: active' || \
    echo \"  ⚠️  NWDAF: \$NWDAF_STATUS\"

echo ''
echo '════════════════════════════════════════'
echo \"DIAGNOSIS COMPLETE — \$ISSUES critical issue(s) found\"
echo '════════════════════════════════════════'
"

════════════════════════════════════════
  OPEN5GS AUTO-DIAGNOSIS
  2026-04-25 22:01:07
════════════════════════════════════════

[1] NF Service Status
  ✅ All NFs active

[2] PFCP Association (SMF ↔ UPF)
  ✅ PFCP OK (1 events)

[3] NAT Masquerade Rules
  ✅ MASQUERADE rule present

[4] IP Forwarding
  ✅ IP forwarding: enabled

[5] MongoDB
  ✅ MongoDB: reachable

[6] gtp5g Kernel Module
  ✅ gtp5g loaded

[7] UE Tunnel Interfaces
  ✅ 2 tunnel interface(s) active

[8] UPF Prometheus Endpoint
  ✅ UPF metrics endpoint reachable

[9] NWDAF Service
  ✅ NWDAF: active

════════════════════════════════════════
DIAGNOSIS COMPLETE — 0 critical issue(s) found
════════════════════════════════════════


## 11.2 — Error Summary (Last 1 Hour)

In [49]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

# Single-quoted --command avoids gcloud mis-parsing Unicode characters (e.g. x
# multiplication sign U+00D7) as CLI arguments.  All awk format strings use
# plain ASCII 'x' as the repeat indicator instead of the Unicode x symbol.
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command='
echo "=== Last 1 Hour Error/Warning Summary ==="
for NF in amfd smfd upfd ausfd udmd pcfd nrfd nwdafd; do
    ERRORS=$(sudo journalctl -u open5gs-$NF \
        --since "1 hour ago" --no-pager 2>/dev/null | \
        grep -cE "ERROR|WARN|error|warn" || true)
    [ "${ERRORS:-0}" -gt 0 ] && printf "  %-12s: %d errors/warnings\n" "$NF" "$ERRORS"
done

echo ""
echo "=== Top Errors --- UPF ==="
sudo journalctl -u open5gs-upfd --since "1 hour ago" --no-pager 2>/dev/null | \
    grep -iE "ERROR|error" | \
    sed "s/.*open5gs-upfd\[.*\]://" | \
    sort | uniq -c | sort -rn | head -10 | \
    awk "{printf \"  [%dx] %s\n\", \$1, substr(\$0, index(\$0,\$2))}"

echo ""
echo "=== Top Errors --- AMF ==="
sudo journalctl -u open5gs-amfd --since "1 hour ago" --no-pager 2>/dev/null | \
    grep -iE "ERROR|error" | \
    sed "s/.*open5gs-amfd\[.*\]://" | \
    sort | uniq -c | sort -rn | head -10 | \
    awk "{printf \"  [%dx] %s\n\", \$1, substr(\$0, index(\$0,\$2))}"

echo ""
echo "=== Top Errors --- NWDAF ==="
sudo journalctl -u open5gs-nwdafd --since "1 hour ago" --no-pager 2>/dev/null | \
    grep -iE "ERROR|error|WARN|warn|failed|Failed" | \
    sed "s/.*open5gs-nwdafd\[.*\]://" | \
    sort | uniq -c | sort -rn | head -10 | \
    awk "{printf \"  [%dx] %s\n\", \$1, substr(\$0, index(\$0,\$2))}"
'

=== Last 1 Hour Error/Warning Summary ===
  ausfd       : 3 errors/warnings
  pcfd        : 9 errors/warnings
  nwdafd      : 87 errors/warnings

=== Top Errors — UPF ===

=== Top Errors — AMF ===


# 12. Repair Runbooks

## 12.1 — PDU Session Stuck Repair
Symptom: upf_sessionnbr stays elevated after UE stops; tunnel interfaces persist.

In [50]:
r'''
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== PDU Session Stuck Repair ==='

    echo 'Before — UPF session count:'
    curl -s http://127.0.0.7:9090/metrics 2>/dev/null | grep 'upf_sessionnbr' | grep -v '#'

    echo 'Stopping UE...'
    sudo pkill -f nr-ue || true
    sleep 3

    echo 'Restarting SMF + UPF...'
    sudo systemctl restart open5gs-smfd; sleep 2
    sudo systemctl restart open5gs-upfd; sleep 5

    echo 'After — UPF session count:'
    curl -s http://127.0.0.7:9090/metrics 2>/dev/null | grep 'upf_sessionnbr' | grep -v '#'

    echo 'Waiting for PFCP re-association...'
    for i in \$(seq 1 10); do
        sleep 1
        PFCP=\$(sudo journalctl -u open5gs-upfd -n 10 --no-pager 2>/dev/null | grep 'PFCP associated')
        [ -n \"\$PFCP\" ] && echo '✅ PFCP OK' && break
        echo \"  \$i/10 waiting...\"
    done

    echo 'Restarting UE...'
    sudo nohup /opt/UERANSIM/build/nr-ue \
        -c /opt/UERANSIM/config/open5gs-ue.yaml > /root/ue.log 2>&1 &
    for i in \$(seq 1 30); do
        sleep 0.5
        IP=\$(ip a show uesimtun0 2>/dev/null | grep 'inet ' | awk '{print \$2}')
        [ -n \"\$IP\" ] && echo '✅ PDU Session restored:' \$IP && break
    done
"

'''

<>:23: SyntaxWarning: invalid escape sequence '\$'
<>:23: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_7723/44074070.py:23: SyntaxWarning: invalid escape sequence '\$'
  for i in \$(seq 1 10); do


'\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    echo \'=== PDU Session Stuck Repair ===\'\n\n    echo \'Before — UPF session count:\'\n    curl -s http://127.0.0.7:9090/metrics 2>/dev/null | grep \'upf_sessionnbr\' | grep -v \'#\'\n\n    echo \'Stopping UE...\'\n    sudo pkill -f nr-ue || true\n    sleep 3\n\n    echo \'Restarting SMF + UPF...\'\n    sudo systemctl restart open5gs-smfd; sleep 2\n    sudo systemctl restart open5gs-upfd; sleep 5\n\n    echo \'After — UPF session count:\'\n    curl -s http://127.0.0.7:9090/metrics 2>/dev/null | grep \'upf_sessionnbr\' | grep -v \'#\'\n\n    echo \'Waiting for PFCP re-association...\'\n    for i in \\$(seq 1 10); do\n        sleep 1\n        PFCP=\\$(sudo journalctl -u open5gs-upfd -n 10 --no-pager 2>/dev/null | grep \'PFCP associated\')\n        [ -n "\\$PFCP" ] && echo \'✅ PFCP OK\' && break\n        echo "  \\$i/10

## 12.2 — NAT / IP Forwarding Repair

Symptom: UE has PDU session but no internet connectivity; pings via nr-binder fail.

In [51]:
'''
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== NAT Repair ==='

    echo 'Current rules:'
    sudo iptables -t nat -L POSTROUTING -n --line-numbers 2>/dev/null

    echo 'Flushing POSTROUTING chain...'
    sudo iptables -t nat -F POSTROUTING

    echo 'Re-applying MASQUERADE rule...'
    sudo iptables -t nat -A POSTROUTING -s 10.45.0.0/16 ! -o ogstun -j MASQUERADE

    echo 'Enabling IP forwarding (persistent)...'
    sudo sysctl -w net.ipv4.ip_forward=1
    grep -q 'net.ipv4.ip_forward' /etc/sysctl.conf || \
        echo 'net.ipv4.ip_forward=1' | sudo tee -a /etc/sysctl.conf

    echo ''
    echo 'Final NAT rules:'
    sudo iptables -t nat -L POSTROUTING -n -v

    echo ''
    echo 'Connectivity test:'
    cd /opt/UERANSIM/build && ./nr-binder uesimtun0 ping -c 3 -q 8.8.8.8 \
        2>&1 | tail -3 || echo 'No UE tunnel — start UE first'
"

'''

'\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    echo \'=== NAT Repair ===\'\n\n    echo \'Current rules:\'\n    sudo iptables -t nat -L POSTROUTING -n --line-numbers 2>/dev/null\n\n    echo \'Flushing POSTROUTING chain...\'\n    sudo iptables -t nat -F POSTROUTING\n\n    echo \'Re-applying MASQUERADE rule...\'\n    sudo iptables -t nat -A POSTROUTING -s 10.45.0.0/16 ! -o ogstun -j MASQUERADE\n\n    echo \'Enabling IP forwarding (persistent)...\'\n    sudo sysctl -w net.ipv4.ip_forward=1\n    grep -q \'net.ipv4.ip_forward\' /etc/sysctl.conf ||         echo \'net.ipv4.ip_forward=1\' | sudo tee -a /etc/sysctl.conf\n\n    echo \'\'\n    echo \'Final NAT rules:\'\n    sudo iptables -t nat -L POSTROUTING -n -v\n\n    echo \'\'\n    echo \'Connectivity test:\'\n    cd /opt/UERANSIM/build && ./nr-binder uesimtun0 ping -c 3 -q 8.8.8.8         2>&1 | tail -3 || echo \'No UE

## 12.3 — gtp5g Kernel Module Reload

Symptom: UPF fails to start; lsmod | grep gtp5g returns nothing.

In [52]:
'''
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    echo '=== gtp5g Kernel Module ==='
    lsmod | grep gtp5g && echo '✅ Loaded' || echo '❌ NOT loaded'

    echo ''
    echo '=== Module Info ==='
    modinfo gtp5g 2>/dev/null | grep -E 'filename|version|description|author'

    echo ''
    echo '=== Reload ==='
    sudo modprobe -r gtp5g 2>/dev/null || true
    sudo modprobe gtp5g && echo '✅ gtp5g reloaded' || echo '❌ Load failed — check dmesg'
    echo ''
    echo '=== Verify ==='
    lsmod | grep gtp5g
"

'''

'\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    echo \'=== gtp5g Kernel Module ===\'\n    lsmod | grep gtp5g && echo \'✅ Loaded\' || echo \'❌ NOT loaded\'\n\n    echo \'\'\n    echo \'=== Module Info ===\'\n    modinfo gtp5g 2>/dev/null | grep -E \'filename|version|description|author\'\n\n    echo \'\'\n    echo \'=== Reload ===\'\n    sudo modprobe -r gtp5g 2>/dev/null || true\n    sudo modprobe gtp5g && echo \'✅ gtp5g reloaded\' || echo \'❌ Load failed — check dmesg\'\n    echo \'\'\n    echo \'=== Verify ===\'\n    lsmod | grep gtp5g\n"\n\n'

## 12.4 — Full System Restart (Nuclear Option)
Use when: Multiple NFs failed, PFCP won't reassociate, or after VM reboot.

In [53]:
r'''

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '=== Full System Restart ==='

echo 'Stopping UERANSIM...'
sudo pkill -f nr-ue  || true
sudo pkill -f nr-gnb || true
sleep 3

echo 'Stopping all Open5GS NFs...'
for svc in nwdafd amfd smfd upfd ausfd udmd pcfd nrfd; do
    sudo systemctl stop open5gs-\$svc 2>/dev/null || true
done
sleep 5

echo 'Starting NFs in dependency order...'
for svc in nrfd udmd ausfd amfd smfd upfd pcfd; do
    sudo systemctl start open5gs-\$svc
    sleep 1
done

echo 'Starting NWDAF...'
sudo systemctl start open5gs-nwdafd 2>/dev/null || true
sleep 2

echo 'Final status:'
for svc in amfd smfd upfd ausfd udmd pcfd nrfd nwdafd; do
    STATUS=\$(systemctl is-active open5gs-\$svc 2>/dev/null || echo 'not-installed')
    printf '  %-12s: %s\n' \"\$svc\" \"\$STATUS\"
done

echo 'Starting gNB...'
sudo nohup /opt/UERANSIM/build/nr-gnb \
    -c /opt/UERANSIM/config/open5gs-gnb.yaml > /root/gnb.log 2>&1 &
sleep 5
pgrep -x nr-gnb && echo '✅ gNB started' || echo '❌ gNB failed'

echo 'Starting UE...'
sudo nohup /opt/UERANSIM/build/nr-ue \
    -c /opt/UERANSIM/config/open5gs-ue.yaml > /root/ue.log 2>&1 &
for i in \$(seq 1 40); do
    sleep 0.5
    IP=\$(ip a show uesimtun0 2>/dev/null | grep 'inet ' | awk '{print \$2}')
    [ -n \"\$IP\" ] && echo '✅ UE up:' \$IP && break
done
"


'''

<>:16: SyntaxWarning: invalid escape sequence '\$'
<>:16: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_7723/3674492781.py:16: SyntaxWarning: invalid escape sequence '\$'
  sudo systemctl stop open5gs-\$svc 2>/dev/null || true


'\n\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\necho \'=== Full System Restart ===\'\n\necho \'Stopping UERANSIM...\'\nsudo pkill -f nr-ue  || true\nsudo pkill -f nr-gnb || true\nsleep 3\n\necho \'Stopping all Open5GS NFs...\'\nfor svc in nwdafd amfd smfd upfd ausfd udmd pcfd nrfd; do\n    sudo systemctl stop open5gs-\\$svc 2>/dev/null || true\ndone\nsleep 5\n\necho \'Starting NFs in dependency order...\'\nfor svc in nrfd udmd ausfd amfd smfd upfd pcfd; do\n    sudo systemctl start open5gs-\\$svc\n    sleep 1\ndone\n\necho \'Starting NWDAF...\'\nsudo systemctl start open5gs-nwdafd 2>/dev/null || true\nsleep 2\n\necho \'Final status:\'\nfor svc in amfd smfd upfd ausfd udmd pcfd nrfd nwdafd; do\n    STATUS=\\$(systemctl is-active open5gs-\\$svc 2>/dev/null || echo \'not-installed\')\n    printf \'  %-12s: %s\n\' "\\$svc" "\\$STATUS"\ndone\n\necho \'Starting gNB...\'\n

# 13. Backup & Restore

## 13.1 — Full Backup (Configs + MongoDB)

In [54]:
'''

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
BACKUP_DATE=$(date +%Y%m%d_%H%M%S)
BACKUP_DIR="backup_${BACKUP_DATE}"
mkdir -p $BACKUP_DIR

echo "=== Open5GS Backup: $BACKUP_DATE ==="

# 1) Config archive
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    sudo tar -czf /tmp/open5gs_configs.tar.gz \
        /etc/open5gs/ /opt/UERANSIM/config/ 2>/dev/null
    echo 'Config archive ready'
"
gcloud compute scp $VM_NAME:/tmp/open5gs_configs.tar.gz \
    $BACKUP_DIR/configs.tar.gz --project=$PROJECT_ID --zone=$ZONE

# 2) MongoDB dump
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    sudo mongodump --db=open5gs --out=/tmp/mongo_backup --quiet 2>/dev/null
    sudo tar -czf /tmp/open5gs_mongodb.tar.gz /tmp/mongo_backup/
    echo 'MongoDB dump ready'
"
gcloud compute scp $VM_NAME:/tmp/open5gs_mongodb.tar.gz \
    $BACKUP_DIR/mongodb.tar.gz --project=$PROJECT_ID --zone=$ZONE

# 3) NWDAF models backup
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    [ -d /opt/nwdaf/nwdaf_models ] && \
    sudo tar -czf /tmp/nwdaf_models.tar.gz /opt/nwdaf/models/ && \
    echo 'NWDAF models archive ready' || echo 'No NWDAF models to backup'
"
gcloud compute scp $VM_NAME:/tmp/nwdaf_models.tar.gz \
    $BACKUP_DIR/nwdaf_models.tar.gz --project=$PROJECT_ID --zone=$ZONE 2>/dev/null || true

echo "✅ Backup complete → $BACKUP_DIR/"
ls -lh $BACKUP_DIR/


'''

'\n\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\nBACKUP_DATE=$(date +%Y%m%d_%H%M%S)\nBACKUP_DIR="backup_${BACKUP_DATE}"\nmkdir -p $BACKUP_DIR\n\necho "=== Open5GS Backup: $BACKUP_DATE ==="\n\n# 1) Config archive\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    sudo tar -czf /tmp/open5gs_configs.tar.gz         /etc/open5gs/ /opt/UERANSIM/config/ 2>/dev/null\n    echo \'Config archive ready\'\n"\ngcloud compute scp $VM_NAME:/tmp/open5gs_configs.tar.gz     $BACKUP_DIR/configs.tar.gz --project=$PROJECT_ID --zone=$ZONE\n\n# 2) MongoDB dump\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    sudo mongodump --db=open5gs --out=/tmp/mongo_backup --quiet 2>/dev/null\n    sudo tar -czf /tmp/open5gs_mongodb.tar.gz /tmp/mongo_backup/\n    echo \'MongoDB dump ready\'\n"\ngcloud compute scp $VM_NAME:/tmp/open5gs_mongodb.tar.gz     $BACKUP_DIR/mongodb.tar.gz --project=$PROJECT_ID --zone=$ZONE\n\n# 3

## 13.2 — Restore from Backup

In [55]:
'''

%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"
BACKUP_DIR="backup_YYYYMMDD_HHMMSS"   # ← set your backup dir

echo "=== Restoring from $BACKUP_DIR ==="

# Upload configs
gcloud compute scp $BACKUP_DIR/configs.tar.gz \
    $VM_NAME:/tmp/configs.tar.gz --project=$PROJECT_ID --zone=$ZONE

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    sudo tar -xzf /tmp/configs.tar.gz -C / 2>/dev/null
    echo 'Configs restored'
"

# Restore MongoDB
gcloud compute scp $BACKUP_DIR/mongodb.tar.gz \
    $VM_NAME:/tmp/mongodb.tar.gz --project=$PROJECT_ID --zone=$ZONE

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    sudo tar -xzf /tmp/mongodb.tar.gz -C /tmp/
    sudo mongorestore --db=open5gs /tmp/mongo_backup/open5gs/ --quiet 2>/dev/null
    echo 'MongoDB restored'
"

echo "✅ Restore complete. Restart Open5GS services (Cell 3.1) to apply."

'''


'\n\n%%bash\nPROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"\nBACKUP_DIR="backup_YYYYMMDD_HHMMSS"   # ← set your backup dir\n\necho "=== Restoring from $BACKUP_DIR ==="\n\n# Upload configs\ngcloud compute scp $BACKUP_DIR/configs.tar.gz     $VM_NAME:/tmp/configs.tar.gz --project=$PROJECT_ID --zone=$ZONE\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    sudo tar -xzf /tmp/configs.tar.gz -C / 2>/dev/null\n    echo \'Configs restored\'\n"\n\n# Restore MongoDB\ngcloud compute scp $BACKUP_DIR/mongodb.tar.gz     $VM_NAME:/tmp/mongodb.tar.gz --project=$PROJECT_ID --zone=$ZONE\n\ngcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="\n    sudo tar -xzf /tmp/mongodb.tar.gz -C /tmp/\n    sudo mongorestore --db=open5gs /tmp/mongo_backup/open5gs/ --quiet 2>/dev/null\n    echo \'MongoDB restored\'\n"\n\necho "✅ Restore complete. Restart Open5GS services (Cell 3.1) to apply."\n\n'

# 15. Quick Reference Card

### Service Names

| NF | systemd service | SBI port | config file |
|---|---|---|---|
| NRF | `open5gs-nrfd` | 127.0.0.10:7777 | `/etc/open5gs/nrf.yaml` |
| AMF | `open5gs-amfd` | 127.0.0.5:7777 | `/etc/open5gs/amf.yaml` |
| SMF | `open5gs-smfd` | 127.0.0.4:7777 | `/etc/open5gs/smf.yaml` |
| UPF | `open5gs-upfd` | 127.0.0.7:9090 (metrics) | `/etc/open5gs/upf.yaml` |
| UDM | `open5gs-udmd` | 127.0.0.12:7777 | `/etc/open5gs/udm.yaml` |
| AUSF | `open5gs-ausfd` | 127.0.0.11:7777 | `/etc/open5gs/ausf.yaml` |
| PCF | `open5gs-pcfd` | 127.0.0.13:7777 | `/etc/open5gs/pcf.yaml` |
| NWDAF | `open5gs-nwdafd` | **127.0.0.1:7779** | `/etc/open5gs/nwdaf.yaml` |

### NWDAF REST Endpoints (TS 29.520)

```
GET    http://localhost:7779/nwdaf-analytics/v1/health
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=NF_LOAD
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=UE_MOBILITY
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=UE_COMMUNICATION
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=ABNORMAL_BEHAVIOUR
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=QoS_SUSTAINABILITY
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=SERVICE_EXPERIENCE
GET    http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=NETWORK_PERFORMANCE
POST   http://localhost:7779/nwdaf-analytics/v1/subscriptions
GET    http://localhost:7779/nwdaf-analytics/v1/subscriptions/{subId}
DELETE http://localhost:7779/nwdaf-analytics/v1/subscriptions/{subId}
```

### Key File Paths (on VM)

| Item | Path |
|---|---|
| Open5GS configs | `/etc/open5gs/*.yaml` |
| UERANSIM configs | `/opt/UERANSIM/config/` |
| UERANSIM binaries | `/opt/UERANSIM/build/` |
| gNB log | `/root/gnb.log` |
| UE log | `/root/ue.log` |
| NWDAF service | `/etc/open5gs/nwdaf.yaml` |
| NWDAF ML models | `/opt/nwdaf/nwdaf_models/` |
| NWDAF systemd unit | `/etc/systemd/system/open5gs-nwdafd.service` |
| GTP tunnel counter (DL) | `/sys/class/net/ogstun/statistics/rx_bytes` |
| GTP tunnel counter (UL) | `/sys/class/net/ogstun/statistics/tx_bytes` |

### Daily Startup Checklist

```
☐ 1. Run Cell 0.1 — GCP auth
☐ 2. Run Cell 0.2 — helper functions
☐ 3. Run Cell 2   — health dashboard → expect 7/7 NFs + gNB + UE + ≥1 PDU session
☐ 4. Run Cell 8.1 — NWDAF health → expect {"status": "UP"}
☐ 5. If anything is red → Cell 11.1 auto-diagnosis → apply repair runbook from §12
☐ 6. If NWDAF was restarted → wait 100s for throughput history to fill before using ABNORMAL_BEHAVIOUR
☐ 7. If ML model is stale → run Cell 9.1 while traffic is flowing

## 13. NWDAF C++ Verification
These cells interact with the C++ NWDAF REST API directly to verify full integration.

In [56]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
echo '=== NWDAF Health Check ==='
curl -s http://localhost:7779/nwdaf-analytics/v1/health | python3 -m json.tool
"


=== NWDAF Health Check ===
{
    "nfInstanceId": "198e9234-b849-42a2-9f70-d9a587616c2b",
    "nfType": "NWDAF",
    "status": "UP",
    "ts": "2026-04-25T22:03:10Z"
}


In [57]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
for ID in NF_LOAD UE_MOBILITY UE_COMMUNICATION ABNORMAL_BEHAVIOUR QoS_SUSTAINABILITY SERVICE_EXPERIENCE NETWORK_PERFORMANCE; do
    echo \"=== \$ID ===\"
    curl -s \"http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=\$ID\" | python3 -m json.tool
    echo \"\"
done
"


=== NF_LOAD ===
{
    "analData": {
        "analyticsId": "NF_LOAD",
        "confidence": 95,
        "nfLoadLevelList": [
            {
                "nfLoadLevelInfo": {
                    "nfCpuUsage": 0.08,
                    "nfLoadLevel": 0.0,
                    "nfLoadLevelLabel": "LOW",
                    "nfMemoryUsage": 34412
                },
                "nfStatus": "active",
                "nfType": "AMF"
            },
            {
                "nfLoadLevelInfo": {
                    "nfCpuUsage": 78.11,
                    "nfLoadLevel": 0.0,
                    "nfLoadLevelLabel": "LOW",
                    "nfMemoryUsage": 34292
                },
                "nfStatus": "active",
                "nfType": "AUSF"
            },
            {
                "nfLoadLevelInfo": {
                    "nfCpuUsage": 77.8,
                    "nfLoadLevel": 0.0,
                    "nfLoadLevelLabel": "LOW",
                    "nfMemoryUsage": 34172
  

## 14. Web UI Integration & Comparison
The NWDAF Operations Dashboard provides a visual representation of the analytics data.
Run the cell below to retrieve the public IP address of the dashboard, check the NWDAF status, and fetch the latest analytics results. You can use these terminal outputs to cross-reference and verify the data displayed in the React web UI.

In [58]:
%%bash
PROJECT_ID="g-ai-lab-491619"; ZONE="europe-west4-a"; VM_NAME="open5gs-ai-lab"

echo '=== Fetching NWDAF Web UI IP Address ==='
EXTERNAL_IP=$(gcloud compute instances describe $VM_NAME \
    --project=$PROJECT_ID \
    --zone=$ZONE \
    --format='get(networkInterfaces[0].accessConfigs[0].natIP)')

echo "🌐 Access the NWDAF Dashboard at: http://$EXTERNAL_IP"
echo "(Note: Ensure the correct port is appended if not running on 80, e.g., http://$EXTERNAL_IP:3000)"
echo ""

echo '=== Current NWDAF Service Status ==='
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    curl -s http://localhost:7779/nwdaf-analytics/v1/health | python3 -m json.tool
"

echo ''
echo '=== Latest Analytics Results for UI Comparison ==='
gcloud compute ssh $VM_NAME --project=$PROJECT_ID --zone=$ZONE --command="
    for ID in NF_LOAD UE_MOBILITY UE_COMMUNICATION ABNORMAL_BEHAVIOUR QoS_SUSTAINABILITY SERVICE_EXPERIENCE NETWORK_PERFORMANCE; do
        echo \"--- \$ID ---\"
        # Print just the analData object to keep the output concise for comparison
        curl -s \"http://localhost:7779/nwdaf-analytics/v1/analytics?analyticsId=\$ID\" | python3 -c '
import sys, json
try:
    d = json.load(sys.stdin)
    print(json.dumps(d.get(\"analData\", d), indent=2))
except:
    pass' || echo 'Failed to fetch'
    done
"


=== Fetching NWDAF Web UI IP Address ===
🌐 Access the NWDAF Dashboard at: http://34.90.43.199
(Note: Ensure the correct port is appended if not running on 80, e.g., http://34.90.43.199:3000)

=== Current NWDAF Service Status ===
{
    "nfInstanceId": "198e9234-b849-42a2-9f70-d9a587616c2b",
    "nfType": "NWDAF",
    "status": "UP",
    "ts": "2026-04-25T22:03:42Z"
}
